In [1]:
import pandas as pd
from pathlib import Path

file_path = Path(r"G:\Shared drives\EO_ECONOMIC ANALYSIS\ea_common\Projects\Training Provider Evaluation Project\ETPLscraper\ETPLscraper\etpl_program_data.csv")

etpl = pd.read_csv(file_path)

print(etpl.shape)
print(etpl.columns.tolist())
print(etpl.head())

(1217, 41)
['url', 'program_name', 'cip_code', 'cip_title', 'program_description', 'creds_offered', 'how_offered', 'when_offered', 'approval', 'address', 'phone_number', 'fax_number', 'training_url', 'last_updated', 'created', 'renewal', 'length_weeks', 'length_hours', 'entrance_requirements', 'school', 'WIOA_approved', 'curriculum_competency_based', 'training_locations', 'WIB', 'type_of_attainment', 'credential_name', 'financial_aid', 'refund_policy', 'program_certified', 'nontraditional_for_women', 'nontraditional_for_men', 'tuition_instate', 'tuition_outstate', 'registration_fee', 'testing_fees', 'book_fees', 'other_fees', 'material_fees', 'total_instate_cost', 'total_outstate_cost', 'date_scraped']
                                                 url  \
0  https://www.azjobconnection.gov/etp/public/ins...   
1  https://www.azjobconnection.gov/etp/public/app...   
2  https://www.azjobconnection.gov/etp/public/ins...   
3  https://www.azjobconnection.gov/etp/public/ins...   
4  https

In [2]:
import pandas as pd
import numpy as np
from pathlib import Path

input_path = Path(r"G:\Shared drives\EO_ECONOMIC ANALYSIS\ea_common\Projects\Training Provider Evaluation Project\ETPLscraper\ETPLscraper\etpl_program_data.csv")

output_path = Path(r"C:\Users\301533\Documents\AI Analysis\Training Programs (ETPL+IPEDS)\etpl_program_data_clean_for_cip_mapping.csv")

etpl = pd.read_csv(input_path)

def clean_cip(x):
    if pd.isna(x):
        return np.nan
    
    x = str(x).replace(".0", "").strip()
    x = x.zfill(6)
    
    return x[:2] + "." + x[2:]

etpl["cip_code_clean"] = etpl["cip_code"].apply(clean_cip)

keep_cols = [
    "program_name",
    "cip_code",
    "cip_code_clean",
    "cip_title",
    "school",
    "WIOA_approved",
    "how_offered",
    "when_offered",
    "length_weeks",
    "length_hours",
    "type_of_attainment",
    "credential_name",
    "training_locations",
    "WIB",
    "tuition_instate",
    "registration_fee",
    "testing_fees",
    "book_fees",
    "other_fees",
    "material_fees",
    "total_instate_cost",
    "url",
    "training_url",
    "date_scraped"
]

etpl_clean = etpl[keep_cols].copy()

etpl_clean.to_csv(output_path, index=False)

print(etpl_clean.shape)
print(etpl_clean.head())
print("Saved to:", output_path)

(1217, 24)
                                 program_name  cip_code cip_code_clean  \
0  Emergency Medical Technology - certificate  519999.0        51.9999   
1          Health Coach Certification Program       NaN            NaN   
2                      Dental Assistant (F2F)  510601.0        51.0601   
3     Commercial Driver License (CDL) Program  490205.0        49.0205   
4             Practical Nursing - certificate  513899.0        51.3899   

                                           cip_title  \
0  Health Professions and Related Clinical Scienc...   
1                                                NaN   
2                        Dental Assisting/Assistant.   
3  Truck and Bus Driver/Commercial Vehicle Operat...   
4  Registered Nursing, Nursing Administration, Nu...   

                             school WIOA_approved        how_offered  \
0                   Cochise College           Yes          In Person   
1  Legacy Holistic Health Institute           NaN              

# Clean ETPL

In [3]:
import pandas as pd
import numpy as np
import re
from pathlib import Path

# =========================================================
# STEP 1: CLEAN ETPL TRAINING PROGRAM FILE
# =========================================================

input_path = Path(
    r"G:\Shared drives\EO_ECONOMIC ANALYSIS\ea_common\Projects\Training Provider Evaluation Project\ETPLscraper\ETPLscraper\etpl_program_data.csv"
)

output_folder = Path(
    r"C:\Users\301533\Documents\AI Analysis\Training Programs (ETPL+IPEDS)"
)

output_folder.mkdir(parents=True, exist_ok=True)

output_csv = output_folder / "ETPL_CLEAN_step1.csv"
output_excel = output_folder / "ETPL_CLEAN_step1.xlsx"

# =========================================================
# LOAD DATA
# =========================================================

etpl = pd.read_csv(input_path)

print("Original shape:", etpl.shape)

# =========================================================
# CLEAN CIP CODE
# Converts 519999.0 to 51.9999
# =========================================================

def clean_cip(x):
    if pd.isna(x):
        return np.nan
    
    x = str(x).strip()
    x = x.replace(".0", "")
    x = re.sub(r"[^0-9]", "", x)
    
    if x == "":
        return np.nan
    
    x = x.zfill(6)
    return x[:2] + "." + x[2:]

etpl["cip_code_clean"] = etpl["cip_code"].apply(clean_cip)

# =========================================================
# CLEAN MONEY FIELDS
# Pulls first dollar amount from messy text fields
# Example: "$1,536.00 Total Credit Hours..." becomes 1536.00
# =========================================================

money_cols = [
    "tuition_instate",
    "tuition_outstate",
    "registration_fee",
    "testing_fees",
    "book_fees",
    "other_fees",
    "material_fees",
    "total_instate_cost",
    "total_outstate_cost"
]

def clean_money(x):
    if pd.isna(x):
        return np.nan
    
    x = str(x)
    match = re.search(r"\$?\s*([0-9,]+(?:\.\d{2})?)", x)
    
    if match:
        return float(match.group(1).replace(",", ""))
    
    return np.nan

for col in money_cols:
    if col in etpl.columns:
        etpl[col + "_numeric"] = etpl[col].apply(clean_money)

etpl_clean["training_source"] = "ETPL"
# =========================================================
# STANDARDIZE YES/NO FIELDS
# =========================================================

def clean_yes_no(x):
    if pd.isna(x):
        return np.nan
    
    x = str(x).strip().lower()
    
    if x in ["yes", "y", "true", "1"]:
        return "Yes"
    elif x in ["no", "n", "false", "0"]:
        return "No"
    else:
        return np.nan

if "WIOA_approved" in etpl.columns:
    etpl["WIOA_approved_clean"] = etpl["WIOA_approved"].apply(clean_yes_no)

if "curriculum_competency_based" in etpl.columns:
    etpl["competency_based_clean"] = etpl["curriculum_competency_based"].apply(clean_yes_no)

# =========================================================
# STANDARDIZE DELIVERY MODE
# =========================================================

def clean_delivery(x):
    if pd.isna(x):
        return np.nan
    
    x = str(x).strip().lower()
    
    if "online" in x or "distance" in x or "e-learning" in x:
        return "Online"
    elif "hybrid" in x or "blended" in x:
        return "Hybrid"
    elif "person" in x:
        return "In Person"
    else:
        return str(x).title()

etpl["delivery_mode_clean"] = etpl["how_offered"].apply(clean_delivery)

# =========================================================
# STANDARDIZE PROGRAM TIMING
# =========================================================

def clean_when_offered(x):
    if pd.isna(x):
        return np.nan
    
    x = str(x).strip()
    
    flags = []
    if "Daytime" in x:
        flags.append("Daytime")
    if "Evening" in x:
        flags.append("Evening")
    if "Weekend" in x:
        flags.append("Weekend")
    
    if len(flags) == 0:
        return x
    
    return "; ".join(flags)

etpl["schedule_clean"] = etpl["when_offered"].apply(clean_when_offered)

# =========================================================
# STANDARDIZE CREDENTIAL CATEGORY
# =========================================================

def clean_credential(row):
    raw_attainment = str(row.get("type_of_attainment", "")).lower()
    raw_credential = str(row.get("credential_name", "")).lower()
    raw_program = str(row.get("program_name", "")).lower()
    
    combined = " ".join([raw_attainment, raw_credential, raw_program])
    
    if "associate" in combined:
        return "Associate Degree"
    elif "bachelor" in combined:
        return "Bachelor Degree"
    elif "master" in combined:
        return "Master Degree"
    elif "doctor" in combined:
        return "Doctorate"
    elif "certificate" in combined:
        return "Certificate"
    elif "certification" in combined or "license" in combined:
        return "Certification or License"
    elif "diploma" in combined:
        return "Diploma"
    elif "apprentice" in combined:
        return "Apprenticeship"
    else:
        return "Other / Unknown"

etpl["credential_category_clean"] = etpl.apply(clean_credential, axis=1)

# =========================================================
# CREATE STANDARDIZED MASTER COLUMNS
# =========================================================

etpl_clean = pd.DataFrame()

etpl_clean["training_source"] = "ETPL"
etpl_clean["program_name"] = etpl["program_name"]
etpl_clean["provider_name"] = etpl["school"]
etpl_clean["cip_code"] = etpl["cip_code_clean"]
etpl_clean["cip_title"] = etpl["cip_title"]
etpl_clean["credential_category"] = etpl["credential_category_clean"]
etpl_clean["credential_name"] = etpl["credential_name"]
etpl_clean["delivery_mode"] = etpl["delivery_mode_clean"]
etpl_clean["schedule"] = etpl["schedule_clean"]
etpl_clean["duration_weeks"] = pd.to_numeric(etpl["length_weeks"], errors="coerce")
etpl_clean["duration_hours"] = pd.to_numeric(etpl["length_hours"], errors="coerce")
etpl_clean["WIOA_approved"] = etpl["WIOA_approved_clean"]
etpl_clean["competency_based"] = etpl["competency_based_clean"]
etpl_clean["training_locations"] = etpl["training_locations"]
etpl_clean["workforce_board"] = etpl["WIB"]
etpl_clean["tuition_instate"] = etpl.get("tuition_instate_numeric")
etpl_clean["total_instate_cost"] = etpl.get("total_instate_cost_numeric")
etpl_clean["program_url"] = etpl["url"]
etpl_clean["training_url"] = etpl["training_url"]
etpl_clean["date_scraped"] = etpl["date_scraped"]

# =========================================================
# BASIC CLEANUP
# =========================================================

for col in etpl_clean.select_dtypes(include="object").columns:
    etpl_clean[col] = etpl_clean[col].apply(
        lambda x: x.strip() if isinstance(x, str) else x
    )

    # =========================================================
# FINAL FIXES BEFORE SAVE
# =========================================================

etpl_clean["training_source"] = "ETPL"

print(etpl_clean["training_source"].value_counts(dropna=False))

# =========================================================
# SAVE FILES
# =========================================================

etpl_clean.to_csv(output_csv, index=False)
etpl_clean.to_excel(output_excel, index=False)

display(etpl_clean.head())
# =========================================================
# SAVE FILES
# =========================================================

etpl_clean.to_csv(output_csv, index=False)
etpl_clean.to_excel(output_excel, index=False)

print("Cleaned ETPL shape:", etpl_clean.shape)
print("Saved CSV to:", output_csv)
print("Saved Excel to:", output_excel)

display(etpl_clean.head())

Original shape: (1217, 41)
training_source
ETPL    1217
Name: count, dtype: int64


,training_source,program_name,provider_name,cip_code,cip_title,credential_category,credential_name,delivery_mode,schedule,duration_weeks,duration_hours,WIOA_approved,competency_based,training_locations,workforce_board,tuition_instate,total_instate_cost,program_url,training_url,date_scraped
0,ETPL,Emergency Medical Technology - certificate,Cochise College,51.9999,Health Professions and Related Clinical Scienc...,Certificate,Emergency Medical Technician Certificate,In Person,Daytime; Evening,16.0,9.0,Yes,Yes,Sierra Vista,1 - ARIZONA@WORK - Southeastern Arizona,864.0,2050.0,https://www.azjobconnection.gov/etp/public/ins...,NaN,2025-05-29
1,ETPL,Health Coach Certification Program,Legacy Holistic Health Institute,NaN,NaN,Certification or License,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,5 - ARIZONA@WORK - City of Phoenix,NaN,NaN,https://www.azjobconnection.gov/etp/public/app...,NaN,2025-05-29
2,ETPL,Dental Assistant (F2F),Cochise College,51.0601,Dental Assisting/Assistant.,Certificate,Dental Assistant Certificate,In Person,Daytime; Evening,16.0,16.0,Yes,NaN,"901 N. Colombo, Sierra Vista, Arizona",1 - ARIZONA@WORK - Southeastern Arizona,1536.0,2376.0,https://www.azjobconnection.gov/etp/public/ins...,https://www.cochise.edu/,2025-05-29
3,ETPL,Commercial Driver License (CDL) Program,Cochise College,49.0205,Truck and Bus Driver/Commercial Vehicle Operat...,Certification or License,"CDL, Class A",Hybrid,Daytime; Weekend,4.0,8.0,Yes,NaN,"901 N. Colombo, Sierra Vista, Arizona",1 - ARIZONA@WORK - Southeastern Arizona,768.0,5496.0,https://www.azjobconnection.gov/etp/public/ins...,http://www.cochise.edu,2025-05-29
4,ETPL,Practical Nursing - certificate,Cochise College,51.3899,"Registered Nursing, Nursing Administration, Nu...",Certificate,Practical Nursing Certificate,In Person,Daytime; Evening,60.0,32.0,Yes,Yes,Sierra Vista,1 - ARIZONA@WORK - Southeastern Arizona,4047.0,8634.0,https://www.azjobconnection.gov/etp/public/ins...,NaN,2025-05-29


Cleaned ETPL shape: (1217, 20)
Saved CSV to: C:\Users\301533\Documents\AI Analysis\Training Programs (ETPL+IPEDS)\ETPL_CLEAN_step1.csv
Saved Excel to: C:\Users\301533\Documents\AI Analysis\Training Programs (ETPL+IPEDS)\ETPL_CLEAN_step1.xlsx


,training_source,program_name,provider_name,cip_code,cip_title,credential_category,credential_name,delivery_mode,schedule,duration_weeks,duration_hours,WIOA_approved,competency_based,training_locations,workforce_board,tuition_instate,total_instate_cost,program_url,training_url,date_scraped
0,ETPL,Emergency Medical Technology - certificate,Cochise College,51.9999,Health Professions and Related Clinical Scienc...,Certificate,Emergency Medical Technician Certificate,In Person,Daytime; Evening,16.0,9.0,Yes,Yes,Sierra Vista,1 - ARIZONA@WORK - Southeastern Arizona,864.0,2050.0,https://www.azjobconnection.gov/etp/public/ins...,NaN,2025-05-29
1,ETPL,Health Coach Certification Program,Legacy Holistic Health Institute,NaN,NaN,Certification or License,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,5 - ARIZONA@WORK - City of Phoenix,NaN,NaN,https://www.azjobconnection.gov/etp/public/app...,NaN,2025-05-29
2,ETPL,Dental Assistant (F2F),Cochise College,51.0601,Dental Assisting/Assistant.,Certificate,Dental Assistant Certificate,In Person,Daytime; Evening,16.0,16.0,Yes,NaN,"901 N. Colombo, Sierra Vista, Arizona",1 - ARIZONA@WORK - Southeastern Arizona,1536.0,2376.0,https://www.azjobconnection.gov/etp/public/ins...,https://www.cochise.edu/,2025-05-29
3,ETPL,Commercial Driver License (CDL) Program,Cochise College,49.0205,Truck and Bus Driver/Commercial Vehicle Operat...,Certification or License,"CDL, Class A",Hybrid,Daytime; Weekend,4.0,8.0,Yes,NaN,"901 N. Colombo, Sierra Vista, Arizona",1 - ARIZONA@WORK - Southeastern Arizona,768.0,5496.0,https://www.azjobconnection.gov/etp/public/ins...,http://www.cochise.edu,2025-05-29
4,ETPL,Practical Nursing - certificate,Cochise College,51.3899,"Registered Nursing, Nursing Administration, Nu...",Certificate,Practical Nursing Certificate,In Person,Daytime; Evening,60.0,32.0,Yes,Yes,Sierra Vista,1 - ARIZONA@WORK - Southeastern Arizona,4047.0,8634.0,https://www.azjobconnection.gov/etp/public/ins...,NaN,2025-05-29


# Step 2: clean IPEDS into the exact same column structure, with training_source = "IPEDS".

In [4]:
import pandas as pd
import numpy as np
import re
from pathlib import Path

# =========================================================
# STEP 2: CLEAN IPEDS TRAINING PROGRAM FILE
# =========================================================

ipeds_input_path = Path(
    r"C:\Users\301533\Documents\AI Training Programs\IPEDS_AZ_CIP_Programs.xlsx"
)

output_folder = Path(
    r"C:\Users\301533\Documents\AI Analysis\Training Programs (ETPL+IPEDS)"
)

output_folder.mkdir(parents=True, exist_ok=True)

output_csv = output_folder / "IPEDS_CLEAN_step2.csv"
output_excel = output_folder / "IPEDS_CLEAN_step2.xlsx"

# =========================================================
# LOAD DATA
# =========================================================

ipeds = pd.read_excel(ipeds_input_path)

print("Original IPEDS shape:", ipeds.shape)
print(ipeds.columns.tolist())

# =========================================================
# IPEDS COLUMN NAMES
# =========================================================

cip_col = "6-Digit CIP Code"
credential_col = "Credential Levels Offered"

program_col = "6-Digit CIP Program Title"
provider_col = "Institution Name"
cip_title_col = "6-Digit CIP Program Title"
city_col = "City"
state_col = "State"
county_col = "County"
website_col = "Institution Website"
application_col = "Application Website"

# =========================================================
# CLEAN CIP CODE
# =========================================================

def clean_cip(x):
    if pd.isna(x):
        return np.nan
    
    x = str(x).strip()
    x = x.replace(".0", "")
    x = re.sub(r"[^0-9]", "", x)

    if x == "":
        return np.nan

    x = x.zfill(6)
    return x[:2] + "." + x[2:]

ipeds["cip_code_clean"] = ipeds[cip_col].apply(clean_cip)

# =========================================================
# STANDARDIZE CREDENTIAL CATEGORY
# =========================================================

def clean_credential(x):
    if pd.isna(x):
        return "Other / Unknown"
    
    x = str(x).lower()

    if "certificate" in x or "cert" in x:
        return "Certificate"
    elif "associate" in x:
        return "Associate Degree"
    elif "bachelor" in x:
        return "Bachelor Degree"
    elif "master" in x:
        return "Master Degree"
    elif "doctor" in x:
        return "Doctorate"
    elif "diploma" in x:
        return "Diploma"
    else:
        return "Other / Unknown"

ipeds["credential_category_clean"] = ipeds[credential_col].apply(clean_credential)

# =========================================================
# HELPER FUNCTION
# =========================================================

def get_col(df, col_name):
    if col_name in df.columns:
        return df[col_name]
    else:
        return pd.Series([np.nan] * len(df))

# =========================================================
# CREATE STANDARDIZED MASTER COLUMNS
# =========================================================

ipeds_clean = pd.DataFrame()

ipeds_clean["training_source"] = "IPEDS"
ipeds_clean["program_name"] = get_col(ipeds, program_col)
ipeds_clean["provider_name"] = get_col(ipeds, provider_col)
ipeds_clean["cip_code"] = ipeds["cip_code_clean"]
ipeds_clean["cip_title"] = get_col(ipeds, cip_title_col)
ipeds_clean["credential_category"] = ipeds["credential_category_clean"]
ipeds_clean["credential_name"] = get_col(ipeds, credential_col)

# Same schema as ETPL
ipeds_clean["delivery_mode"] = np.nan
ipeds_clean["schedule"] = np.nan
ipeds_clean["duration_weeks"] = np.nan
ipeds_clean["duration_hours"] = np.nan
ipeds_clean["WIOA_approved"] = np.nan
ipeds_clean["competency_based"] = np.nan

ipeds_clean["training_locations"] = (
    get_col(ipeds, city_col).astype(str).replace("nan", "") + ", " +
    get_col(ipeds, state_col).astype(str).replace("nan", "")
)

ipeds_clean["workforce_board"] = np.nan
ipeds_clean["tuition_instate"] = np.nan
ipeds_clean["total_instate_cost"] = np.nan

ipeds_clean["program_url"] = get_col(ipeds, application_col)
ipeds_clean["training_url"] = get_col(ipeds, website_col)
ipeds_clean["date_scraped"] = np.nan

# Extra IPEDS-only fields
ipeds_clean["city"] = get_col(ipeds, city_col)
ipeds_clean["county"] = get_col(ipeds, county_col)
ipeds_clean["state"] = get_col(ipeds, state_col)

# =========================================================
# CLEAN TEXT FIELDS
# =========================================================

for col in ipeds_clean.select_dtypes(include="object").columns:
    ipeds_clean[col] = ipeds_clean[col].apply(
        lambda x: x.strip() if isinstance(x, str) else x
    )
    ipeds_clean[col] = ipeds_clean[col].replace(
        {"": np.nan, "nan,": np.nan, ",": np.nan}
    )

ipeds_clean["training_source"] = "IPEDS"

# =========================================================
# SAVE FILES
# =========================================================

ipeds_clean.to_csv(output_csv, index=False)
ipeds_clean.to_excel(output_excel, index=False)

print("Cleaned IPEDS shape:", ipeds_clean.shape)
print("Saved CSV to:", output_csv)
print("Saved Excel to:", output_excel)

display(ipeds_clean.head())

Original IPEDS shape: (2620, 16)
['IPEDS Unit ID', 'Institution Name', 'Year', '6-Digit CIP Code', '6-Digit CIP Program Title', 'Total Completions', 'Credential Levels Offered', 'Street Address', 'City', 'State', 'ZIP Code', 'County', 'Institution Sector', 'Institution Level', 'Institution Website', 'Application Website']
Cleaned IPEDS shape: (2620, 23)
Saved CSV to: C:\Users\301533\Documents\AI Analysis\Training Programs (ETPL+IPEDS)\IPEDS_CLEAN_step2.csv
Saved Excel to: C:\Users\301533\Documents\AI Analysis\Training Programs (ETPL+IPEDS)\IPEDS_CLEAN_step2.xlsx


,training_source,program_name,provider_name,cip_code,cip_title,credential_category,credential_name,delivery_mode,schedule,duration_weeks,...,training_locations,workforce_board,tuition_instate,total_instate_cost,program_url,training_url,date_scraped,city,county,state
0,IPEDS,Medical/Clinical Assistant,Allen School-Phoenix,05.1801,Medical/Clinical Assistant,Certificate,Certificate <1 year,NaN,NaN,NaN,...,"Phoenix, Arizona",NaN,NaN,NaN,www.allenschool.edu/,www.allenschool.edu/,NaN,Phoenix,Maricopa County,Arizona
1,IPEDS,"Education, General",American InterContinental University System,01.3101,"Education, General",Master Degree,Master's degree,NaN,NaN,NaN,...,"Chandler, Arizona",NaN,NaN,NaN,https://www.aius.education/about-aius/institut...,https://www.aius.education/,NaN,Chandler,Maricopa County,Arizona
2,IPEDS,"Educational Leadership and Administration, Gen...",American InterContinental University System,01.3401,"Educational Leadership and Administration, Gen...",Doctorate,Doctoral degree,NaN,NaN,NaN,...,"Chandler, Arizona",NaN,NaN,NaN,https://www.aius.education/about-aius/institut...,https://www.aius.education/,NaN,Chandler,Maricopa County,Arizona
3,IPEDS,Educational/Instructional Technology,American InterContinental University System,01.3501,Educational/Instructional Technology,Master Degree,Master's degree,NaN,NaN,NaN,...,"Chandler, Arizona",NaN,NaN,NaN,https://www.aius.education/about-aius/institut...,https://www.aius.education/,NaN,Chandler,Maricopa County,Arizona
4,IPEDS,"Teacher Education, Multiple Levels",American InterContinental University System,13.1206,"Teacher Education, Multiple Levels",Master Degree,Master's degree,NaN,NaN,NaN,...,"Chandler, Arizona",NaN,NaN,NaN,https://www.aius.education/about-aius/institut...,https://www.aius.education/,NaN,Chandler,Maricopa County,Arizona


# Step 3A: Clean / Repair CIP Codes.

In [5]:
import pandas as pd
import numpy as np
from pathlib import Path

# =========================================================
# STEP 3A: REPAIR / IMPUTE MISSING CIP CODES
# =========================================================

folder = Path(r"C:\Users\301533\Documents\AI Analysis\Training Programs (ETPL+IPEDS)")

etpl_path = folder / "ETPL_CLEAN_step1.csv"
ipeds_path = folder / "IPEDS_CLEAN_step2.csv"

output_csv = folder / "TRAINING_PROGRAMS_MASTER_step3A_cip_cleaned.csv"
output_excel = folder / "TRAINING_PROGRAMS_MASTER_step3A_cip_cleaned.xlsx"
flags_excel = folder / "CIP_REPAIR_FLAGS_step3A.xlsx"

# =========================================================
# LOAD FILES
# =========================================================

etpl = pd.read_csv(etpl_path, dtype={"cip_code": str})
ipeds = pd.read_csv(ipeds_path, dtype={"cip_code": str})

training = pd.concat([etpl, ipeds], ignore_index=True)

print("Combined shape:", training.shape)
print(training["training_source"].value_counts(dropna=False))

# =========================================================
# PRESERVE ORIGINAL CIP
# =========================================================

training["cip_code_original"] = training["cip_code"]
training["cip_imputation_method"] = np.nan
training["cip_code_imputed"] = np.nan

# =========================================================
# KEYWORD-BASED CIP IMPUTATION
# Only used when CIP is missing
# =========================================================

def impute_cip(row):
    cip = row.get("cip_code")
    program = str(row.get("program_name", "")).lower()
    title = str(row.get("cip_title", "")).lower()
    cred = str(row.get("credential_name", "")).lower()
    provider = str(row.get("provider_name", "")).lower()
    
    text = " ".join([program, title, cred, provider])
    
    if pd.notna(cip) and str(cip).strip() != "":
        return pd.Series([np.nan, np.nan])
    
    # Health
    if "medical assistant" in text or "medical/clinical assistant" in text:
        return pd.Series(["51.0801", "keyword: medical assistant"])
    if "nursing assistant" in text or "cna" in text:
        return pd.Series(["51.3902", "keyword: nursing assistant / CNA"])
    if "practical nursing" in text or "lpn" in text:
        return pd.Series(["51.3901", "keyword: practical nursing / LPN"])
    if "registered nursing" in text or "rn" in text:
        return pd.Series(["51.3801", "keyword: registered nursing"])
    if "dental assistant" in text:
        return pd.Series(["51.0601", "keyword: dental assistant"])
    if "emergency medical" in text or "emt" in text or "paramedic" in text:
        return pd.Series(["51.0904", "keyword: EMT / paramedic"])
    if "health coach" in text:
        return pd.Series(["51.0000", "keyword: health coach broad health"])
    
    # IT / computer
    if "cybersecurity" in text or "security+" in text:
        return pd.Series(["11.1003", "keyword: cybersecurity"])
    if "data analyst" in text or "data analytics" in text:
        return pd.Series(["11.0802", "keyword: data analytics"])
    if "help desk" in text or "computer support" in text or "support specialist" in text:
        return pd.Series(["11.1006", "keyword: computer support"])
    if "application developer" in text or "software developer" in text or "web developer" in text:
        return pd.Series(["11.0201", "keyword: application/software developer"])
    if "network+" in text or "network technician" in text:
        return pd.Series(["11.1001", "keyword: network technician"])
    if "comptia" in text:
        return pd.Series(["11.0101", "keyword: general IT / CompTIA"])
    
    # Construction / trades
    if "electrical" in text or "electrician" in text or "wireman" in text:
        return pd.Series(["46.0302", "keyword: electrician"])
    if "hvac" in text or "heating" in text or "air conditioning" in text:
        return pd.Series(["47.0201", "keyword: HVAC"])
    if "plumbing" in text or "plumber" in text:
        return pd.Series(["46.0503", "keyword: plumbing"])
    if "carpentry" in text or "carpenter" in text:
        return pd.Series(["46.0201", "keyword: carpentry"])
    if "construction" in text:
        return pd.Series(["46.0000", "keyword: construction broad"])
    if "crane operator" in text:
        return pd.Series(["49.0202", "keyword: crane/heavy equipment operator"])
    
    # Transportation / mechanics
    if "cdl" in text or "commercial driver" in text or "truck driver" in text:
        return pd.Series(["49.0205", "keyword: CDL / truck driving"])
    if "airframe" in text or "aviation mechanic" in text:
        return pd.Series(["47.0607", "keyword: airframe / aviation mechanic"])
    if "automotive" in text or "auto mechanic" in text:
        return pd.Series(["47.0604", "keyword: automotive"])
    if "diesel" in text:
        return pd.Series(["47.0613", "keyword: diesel mechanic"])
    if "industrial maintenance" in text or "maintenance mechanic" in text:
        return pd.Series(["47.0303", "keyword: industrial maintenance"])
    
    # Personal services
    if "cosmetology" in text:
        return pd.Series(["12.0401", "keyword: cosmetology"])
    if "esthetician" in text or "aesthetician" in text:
        return pd.Series(["12.0409", "keyword: esthetician"])
    if "barber" in text:
        return pd.Series(["12.0402", "keyword: barber"])
    
    # Business / admin
    if "project manager" in text or "project management" in text or "pmp" in text or "capm" in text:
        return pd.Series(["52.0211", "keyword: project management"])
    if "accounting" in text or "bookkeeping" in text:
        return pd.Series(["52.0302", "keyword: accounting / bookkeeping"])
    if "human resources" in text or "hr " in text:
        return pd.Series(["52.1001", "keyword: human resources"])
    if "technical sales" in text or "sales representative" in text:
        return pd.Series(["52.1804", "keyword: selling skills / sales"])
    if "digital marketer" in text or "marketing" in text:
        return pd.Series(["52.1401", "keyword: marketing"])
    
    # Legal
    if "paralegal" in text or "legal assistant" in text:
        return pd.Series(["22.0302", "keyword: paralegal"])
    
    # Culinary
    if "culinary" in text or "food group" in text or "rouxbe" in text:
        return pd.Series(["12.0503", "keyword: culinary arts"])
    
    # Broadband / fiber
    if "fiber optic" in text or "broadband" in text:
        return pd.Series(["15.0303", "keyword: telecommunications technology"])
    
    return pd.Series([np.nan, np.nan])

training[["cip_code_imputed", "cip_imputation_method"]] = training.apply(impute_cip, axis=1)

# =========================================================
# MANUAL CIP OVERRIDES
# Only for high-confidence matches
# =========================================================

manual_cip_map = {
    "Application Development and Support": "11.0201",
    "Computer and Information Technology": "11.0101",
    "Cyber Security Support Technician": "11.1003",
    "End User Computing": "11.1006",
    "IT Infrastructure Support": "11.1006",
    "Information Technology Specialist": "11.0101",
    "Network and computer systems administrator": "11.1001",
    "Software Engineer Apprenticeship": "11.0201",
    "Tech Project Coordinator Apprenticeship": "52.0211",
    "ASU Community Health Worker Training Program": "51.2211",
    "Hair Design": "12.0407",
    "Manicurist / Nail Technician": "12.0410",
    "Software Testing Services": "11.0201",
    "Fire Sprinkler Fitter Apprenticeship": "46.0502",
    "AMCA Central AZ Masonry Apprenticeship Program": "46.0101",
    "Phoenix Sheet Metal Joint Apprenticeship and Training Program": "46.0501",
    "Pipefitter - Refrigeration": "46.0506",
    "Steamfitter": "46.0502",
}

training["cip_code_manual"] = training["program_name"].map(manual_cip_map)

training["cip_manual_method"] = ""

training.loc[
    training["cip_code_manual"].notna(),
    "cip_manual_method"
] = "manual high-confidence program-name match"

training["cip_manual_method"] = training["cip_manual_method"].replace("", np.nan)

# =========================================================
# FINAL CIP CODE
# Priority:
# 1. original CIP
# 2. keyword-imputed CIP
# 3. manual CIP
# =========================================================

training["cip_code_final"] = training["cip_code_original"]

training.loc[
    training["cip_code_final"].isna() & training["cip_code_imputed"].notna(),
    "cip_code_final"
] = training["cip_code_imputed"]

training.loc[
    training["cip_code_final"].isna() & training["cip_code_manual"].notna(),
    "cip_code_final"
] = training["cip_code_manual"]

training["cip_was_imputed"] = (
    training["cip_code_original"].isna() &
    training["cip_code_imputed"].notna()
)

training["cip_was_manual"] = (
    training["cip_code_original"].isna() &
    training["cip_code_imputed"].isna() &
    training["cip_code_manual"].notna()
)

training["cip_still_missing"] = training["cip_code_final"].isna()

# =========================================================
# BASIC FORMAT CHECK
# =========================================================

training["cip_code_final_valid"] = training["cip_code_final"].str.match(
    r"^\d{2}\.\d{4}$", na=False
)

# =========================================================
# SUMMARY
# =========================================================

print("\nCIP repair summary:")
print("Original missing CIP:", training["cip_code_original"].isna().sum())
print("Keyword-imputed CIP:", training["cip_was_imputed"].sum())
print("Manual CIP:", training["cip_was_manual"].sum())
print("Still missing CIP:", training["cip_still_missing"].sum())
print("Invalid final CIP format:", (~training["cip_code_final_valid"]).sum())

print("\nKeyword imputation methods:")
print(training["cip_imputation_method"].value_counts(dropna=True).head(30))

print("\nManual CIP methods:")
print(training["cip_manual_method"].value_counts(dropna=True))

# =========================================================
# SAVE FLAGS
# =========================================================

keyword_imputed = training[training["cip_was_imputed"]].copy()
manual_imputed = training[training["cip_was_manual"]].copy()
still_missing = training[training["cip_still_missing"]].copy()
invalid_final = training[~training["cip_code_final_valid"]].copy()

with pd.ExcelWriter(flags_excel, engine="openpyxl") as writer:
    keyword_imputed.to_excel(writer, sheet_name="Keyword_Imputed_CIP", index=False)
    manual_imputed.to_excel(writer, sheet_name="Manual_CIP", index=False)
    still_missing.to_excel(writer, sheet_name="Still_Missing_CIP", index=False)
    invalid_final.to_excel(writer, sheet_name="Invalid_Final_CIP", index=False)

# =========================================================
# SAVE MASTER FILE
# =========================================================

training.to_csv(output_csv, index=False)
training.to_excel(output_excel, index=False)

print("\nSaved cleaned master file to:")
print(output_csv)
print(output_excel)

display(training[[
    "training_source",
    "program_name",
    "provider_name",
    "cip_code_original",
    "cip_code_imputed",
    "cip_code_manual",
    "cip_code_final",
    "cip_imputation_method",
    "cip_manual_method"
]].head(25))

Combined shape: (3837, 23)
training_source
IPEDS    2620
ETPL     1217
Name: count, dtype: int64

CIP repair summary:
Original missing CIP: 133
Keyword-imputed CIP: 66
Manual CIP: 19
Still missing CIP: 48
Invalid final CIP format: 48

Keyword imputation methods:
cip_imputation_method
keyword: registered nursing                10
keyword: automotive                         8
keyword: electrician                        6
keyword: application/software developer     4
keyword: construction broad                 4
keyword: computer support                   4
keyword: cosmetology                        3
keyword: culinary arts                      3
keyword: barber                             3
keyword: HVAC                               2
keyword: carpentry                          2
keyword: plumbing                           2
keyword: esthetician                        2
keyword: medical assistant                  2
keyword: crane/heavy equipment operator     1
keyword: health coach bro

,training_source,program_name,provider_name,cip_code_original,cip_code_imputed,cip_code_manual,cip_code_final,cip_imputation_method,cip_manual_method
0,ETPL,Emergency Medical Technology - certificate,Cochise College,51.9999,NaN,NaN,51.9999,NaN,NaN
1,ETPL,Health Coach Certification Program,Legacy Holistic Health Institute,NaN,51.0000,NaN,51.0000,keyword: health coach broad health,NaN
2,ETPL,Dental Assistant (F2F),Cochise College,51.0601,NaN,NaN,51.0601,NaN,NaN
3,ETPL,Commercial Driver License (CDL) Program,Cochise College,49.0205,NaN,NaN,49.0205,NaN,NaN
4,ETPL,Practical Nursing - certificate,Cochise College,51.3899,NaN,NaN,51.3899,NaN,NaN
5,ETPL,CompTIA TECH+,Maricopa Corporate College,11.0701,NaN,NaN,11.0701,NaN,NaN
6,ETPL,CompTIA TECH+ & A+,Maricopa Corporate College,11.0101,NaN,NaN,11.0101,NaN,NaN
7,ETPL,Project Manager with CAPM® and PMP® Prep,Maricopa Corporate College,52.0211,NaN,NaN,52.0211,NaN,NaN
8,ETPL,CompTIA A+,Maricopa Corporate College,11.0101,NaN,NaN,11.0101,NaN,NaN
9,ETPL,CompTIA Network+ & Security+,Maricopa Corporate College,11.1003,NaN,NaN,11.1003,NaN,NaN


In [6]:
import pandas as pd
from pathlib import Path

folder = Path(r"C:\Users\301533\Documents\AI Analysis\Training Programs (ETPL+IPEDS)")

master_path = folder / "TRAINING_PROGRAMS_MASTER_step3A_cip_cleaned.xlsx"

training = pd.read_excel(master_path)

# =========================================================
# KEEP ONLY STILL-MISSING CIP PROGRAMS
# =========================================================

missing = training[
    training["cip_still_missing"] == True
].copy()

# =========================================================
# SELECT REVIEW COLUMNS
# =========================================================

review_cols = [
    "training_source",
    "program_name",
    "provider_name",
    "credential_category",
    "credential_name",
    "training_locations",
    "delivery_mode",
    "cip_code_original",
    "cip_code_imputed",
    "cip_code_final"
]

missing_review = missing[review_cols].copy()

# =========================================================
# ADD MANUAL REVIEW COLUMNS
# =========================================================

missing_review["manual_cip_code"] = ""
missing_review["manual_notes"] = ""

# =========================================================
# SORT
# =========================================================

missing_review = missing_review.sort_values(
    by=["provider_name", "program_name"]
)

# =========================================================
# SAVE REVIEW FILE
# =========================================================

review_output = folder / "MANUAL_CIP_REVIEW.xlsx"

missing_review.to_excel(review_output, index=False)

print("Still missing CIP:", missing_review.shape[0])
print("Saved review file to:")
print(review_output)

display(missing_review.head(25))

Still missing CIP: 48
Saved review file to:
C:\Users\301533\Documents\AI Analysis\Training Programs (ETPL+IPEDS)\MANUAL_CIP_REVIEW.xlsx


,training_source,program_name,provider_name,credential_category,credential_name,training_locations,delivery_mode,cip_code_original,cip_code_imputed,cip_code_final,manual_cip_code,manual_notes
978,ETPL,Application Support,Apprentice Now,Other / Unknown,NaN,NaN,NaN,NaN,NaN,NaN,,
1215,ETPL,"Arizona Chapter, Associated General Contractors","Arizona Chapter, Associated General Contractor...",Associate Degree,NaN,NaN,NaN,NaN,NaN,NaN,,
947,ETPL,Arizona Heat and Frost Insulators and Allied W...,Arizona Heat & Frost Insulators and Allied Wor...,Other / Unknown,NaN,NaN,NaN,NaN,NaN,NaN,,
945,ETPL,COP Parks and Recreation Dept and LIUNA Local 777,Arizona Landscapers Contractors Assoc,Other / Unknown,NaN,"4341 Broadway Road, Phoenix Arizona",NaN,NaN,NaN,NaN,,
1117,ETPL,Phoenix Sheet Metal Joint Apprenticeship and T...,Arizona Sheet Metal Joint Apprenticeship and T...,Apprenticeship,NaN,NaN,NaN,NaN,NaN,NaN,,
1025,ETPL,Emerging Technology Apprenticeship Program,"Automation Strategy Performance, Inc. (ASP)",Apprenticeship,NaN,NaN,NaN,NaN,NaN,NaN,,
1203,ETPL,Az Operating Engineers Apprenticeship Training...,Az Operating Engineers Apprenticeship & Traini...,Apprenticeship,NaN,NaN,NaN,NaN,NaN,NaN,,
1024,ETPL,PBM Pharmacy Technician Apprenticeship Program,CVS Health,Apprenticeship,NaN,NaN,NaN,NaN,NaN,NaN,,
1129,ETPL,Carlson Glass Glazier Apprenticeship Program,Carlson Glass,Apprenticeship,NaN,NaN,NaN,NaN,NaN,NaN,,
182,ETPL,DPUniversity,DPUniversity,Other / Unknown,NaN,"Tempe, AZ",NaN,NaN,NaN,NaN,,


# Step 3B: Create Final Master Training Program File.

In [7]:
import pandas as pd
import numpy as np
from pathlib import Path

# =========================================================
# STEP 3B: CREATE FINAL MASTER TRAINING PROGRAM FILE
# =========================================================

folder = Path(r"C:\Users\301533\Documents\AI Analysis\Training Programs (ETPL+IPEDS)")

input_path = folder / "TRAINING_PROGRAMS_MASTER_step3A_cip_cleaned.xlsx"

output_csv = folder / "TRAINING_PROGRAMS_MASTER_FINAL.csv"
output_excel = folder / "TRAINING_PROGRAMS_MASTER_FINAL.xlsx"

summary_excel = folder / "TRAINING_PROGRAM_SUMMARY.xlsx"

# =========================================================
# LOAD DATA
# =========================================================

training = pd.read_excel(input_path, dtype={"cip_code_final": str})

print("Loaded shape:", training.shape)

# =========================================================
# CREATE FINAL CLEAN CIP FIELD
# =========================================================

training["cip_code"] = training["cip_code_final"]

training["cip_2digit"] = training["cip_code"].str[:2]

# =========================================================
# STANDARDIZE TEXT FIELDS
# =========================================================

text_cols = [
    "program_name",
    "provider_name",
    "cip_title",
    "credential_category",
    "credential_name",
    "delivery_mode",
    "training_locations",
    "workforce_board"
]

for col in text_cols:
    if col in training.columns:
        training[col] = (
            training[col]
            .astype(str)
            .str.strip()
            .replace({
                "nan": np.nan,
                "None": np.nan,
                "": np.nan
            })
        )

# =========================================================
# CREATE PROGRAM UID
# =========================================================

training["program_uid"] = (
    training["training_source"].astype(str) + "_" +
    training["provider_name"].astype(str) + "_" +
    training["program_name"].astype(str)
)

# =========================================================
# REMOVE EXACT DUPLICATES
# =========================================================

before = training.shape[0]

training = training.drop_duplicates(
    subset=[
        "training_source",
        "provider_name",
        "program_name",
        "cip_code"
    ]
)

after = training.shape[0]

print("\nDuplicates removed:", before - after)

# =========================================================
# CREATE ANALYSIS FLAGS
# =========================================================

training["has_valid_cip"] = training["cip_code"].notna()

training["is_etpl"] = training["training_source"] == "ETPL"
training["is_ipeds"] = training["training_source"] == "IPEDS"

training["has_cost_data"] = training["total_instate_cost"].notna()

training["has_wioa"] = (
    training["WIOA_approved"]
    .astype(str)
    .str.lower()
    .eq("yes")
)

training["is_online"] = (
    training["delivery_mode"]
    .astype(str)
    .str.contains("online", case=False, na=False)
)

training["is_hybrid"] = (
    training["delivery_mode"]
    .astype(str)
    .str.contains("hybrid", case=False, na=False)
)

training["is_in_person"] = (
    training["delivery_mode"]
    .astype(str)
    .str.contains("person", case=False, na=False)
)

# =========================================================
# CREATE SUMMARY TABLES
# =========================================================

summary_by_source = (
    training.groupby("training_source")
    .agg(
        programs=("program_uid", "count"),
        providers=("provider_name", "nunique"),
        valid_cip=("has_valid_cip", "sum")
    )
    .reset_index()
)

summary_by_credential = (
    training.groupby("credential_category")
    .agg(
        programs=("program_uid", "count")
    )
    .reset_index()
    .sort_values(by="programs", ascending=False)
)

summary_by_cip2 = (
    training.groupby("cip_2digit")
    .agg(
        programs=("program_uid", "count"),
        providers=("provider_name", "nunique")
    )
    .reset_index()
    .sort_values(by="programs", ascending=False)
)

# =========================================================
# SAVE SUMMARY FILE
# =========================================================

with pd.ExcelWriter(summary_excel, engine="openpyxl") as writer:
    summary_by_source.to_excel(writer, sheet_name="By_Source", index=False)
    summary_by_credential.to_excel(writer, sheet_name="By_Credential", index=False)
    summary_by_cip2.to_excel(writer, sheet_name="By_CIP2", index=False)

# =========================================================
# SAVE FINAL MASTER FILE
# =========================================================

training.to_csv(output_csv, index=False)
training.to_excel(output_excel, index=False)

# =========================================================
# FINAL SUMMARY
# =========================================================

print("\nFINAL MASTER FILE SUMMARY")
print("=" * 50)

print("Final shape:", training.shape)

print("\nTraining sources:")
print(training["training_source"].value_counts(dropna=False))

print("\nCredential categories:")
print(training["credential_category"].value_counts(dropna=False).head(15))

print("\nPrograms with valid CIP:")
print(training["has_valid_cip"].sum())

print("\nPrograms still missing CIP:")
print((~training["has_valid_cip"]).sum())

print("\nUnique providers:")
print(training["provider_name"].nunique())

print("\nUnique CIP codes:")
print(training["cip_code"].nunique())

print("\nSaved final master file to:")
print(output_csv)
print(output_excel)

print("\nSaved summary workbook to:")
print(summary_excel)

display(training.head(25))

Loaded shape: (3837, 33)

Duplicates removed: 9

FINAL MASTER FILE SUMMARY
Final shape: (3828, 43)

Training sources:
training_source
IPEDS    2619
ETPL     1209
Name: count, dtype: int64

Credential categories:
credential_category
Certificate                 1689
Associate Degree             757
Bachelor Degree              553
Other / Unknown              311
Master Degree                218
Certification or License     177
Apprenticeship                54
Doctorate                     50
Diploma                       19
Name: count, dtype: int64

Programs with valid CIP:
3781

Programs still missing CIP:
47

Unique providers:
310

Unique CIP codes:
943

Saved final master file to:
C:\Users\301533\Documents\AI Analysis\Training Programs (ETPL+IPEDS)\TRAINING_PROGRAMS_MASTER_FINAL.csv
C:\Users\301533\Documents\AI Analysis\Training Programs (ETPL+IPEDS)\TRAINING_PROGRAMS_MASTER_FINAL.xlsx

Saved summary workbook to:
C:\Users\301533\Documents\AI Analysis\Training Programs (ETPL+IPEDS)\T

,training_source,program_name,provider_name,cip_code,cip_title,credential_category,credential_name,delivery_mode,schedule,duration_weeks,...,cip_2digit,program_uid,has_valid_cip,is_etpl,is_ipeds,has_cost_data,has_wioa,is_online,is_hybrid,is_in_person
0,ETPL,Emergency Medical Technology - certificate,Cochise College,51.9999,Health Professions and Related Clinical Scienc...,Certificate,Emergency Medical Technician Certificate,In Person,Daytime; Evening,16.0,...,51,ETPL_Cochise College_Emergency Medical Technol...,True,True,False,True,True,False,False,True
1,ETPL,Health Coach Certification Program,Legacy Holistic Health Institute,51.0000,NaN,Certification or License,NaN,NaN,NaN,NaN,...,51,ETPL_Legacy Holistic Health Institute_Health C...,True,True,False,False,False,False,False,False
2,ETPL,Dental Assistant (F2F),Cochise College,51.0601,Dental Assisting/Assistant.,Certificate,Dental Assistant Certificate,In Person,Daytime; Evening,16.0,...,51,ETPL_Cochise College_Dental Assistant (F2F),True,True,False,True,True,False,False,True
3,ETPL,Commercial Driver License (CDL) Program,Cochise College,49.0205,Truck and Bus Driver/Commercial Vehicle Operat...,Certification or License,"CDL, Class A",Hybrid,Daytime; Weekend,4.0,...,49,ETPL_Cochise College_Commercial Driver License...,True,True,False,True,True,False,True,False
4,ETPL,Practical Nursing - certificate,Cochise College,51.3899,"Registered Nursing, Nursing Administration, Nu...",Certificate,Practical Nursing Certificate,In Person,Daytime; Evening,60.0,...,51,ETPL_Cochise College_Practical Nursing - certi...,True,True,False,True,True,False,False,True
5,ETPL,CompTIA TECH+,Maricopa Corporate College,11.0701,Computer Science.,Other / Unknown,CompTIA TECH+,Online,Daytime; Evening; Weekend,26.0,...,11,ETPL_Maricopa Corporate College_CompTIA TECH+,True,True,False,True,True,True,False,False
6,ETPL,CompTIA TECH+ & A+,Maricopa Corporate College,11.0101,"Computer and Information Sciences, General.",Other / Unknown,CompTIA TECH+ and A+,Online,Daytime; Evening; Weekend,39.0,...,11,ETPL_Maricopa Corporate College_CompTIA TECH+ ...,True,True,False,True,True,True,False,False
7,ETPL,Project Manager with CAPM® and PMP® Prep,Maricopa Corporate College,52.0211,Project Management.,Other / Unknown,CAPM® and PMP®,Online,Daytime; Evening; Weekend,52.0,...,52,ETPL_Maricopa Corporate College_Project Manage...,True,True,False,True,True,True,False,False
8,ETPL,CompTIA A+,Maricopa Corporate College,11.0101,"Computer and Information Sciences, General.",Other / Unknown,CompTIA A+,Online,Daytime; Evening; Weekend,26.0,...,11,ETPL_Maricopa Corporate College_CompTIA A+,True,True,False,True,True,True,False,False
9,ETPL,CompTIA Network+ & Security+,Maricopa Corporate College,11.1003,Computer and Information Systems Security/Audi...,Other / Unknown,CompTIA Network+ and Security+,Online,Daytime; Evening; Weekend,39.0,...,11,ETPL_Maricopa Corporate College_CompTIA Networ...,True,True,False,True,True,True,False,False


# Step 4A: Inspect Related Occupations File.

In [8]:
import pandas as pd
from pathlib import Path

# =========================================================
# STEP 4A: INSPECT RELATED OCCUPATIONS WORKBOOK
# =========================================================

related_path = Path(
    r"G:\Shared drives\EO_ECONOMIC ANALYSIS\ea_common\Projects\AI\Related Occupations\High_VeryHigh_AND_Medium_Related_Occs_Final.xlsx"
)

# =========================================================
# LOAD WORKBOOK
# =========================================================

xls = pd.ExcelFile(related_path)

print("Sheet names:")
print(xls.sheet_names)

# =========================================================
# PREVIEW EACH SHEET
# =========================================================

for sheet in xls.sheet_names:

    print("\n" + "=" * 80)
    print("SHEET:", sheet)
    print("=" * 80)

    df = pd.read_excel(related_path, sheet_name=sheet)

    print("Shape:", df.shape)

    print("\nColumns:")
    print(df.columns.tolist())

    print("\nFirst 5 rows:")
    display(df.head())

Sheet names:
['Related_Occs_Final', 'Final_Transitions', 'Clean_Direct_Lower_Only', 'Clean_Gateway_Pathways', 'Lightcast_Parsed', 'ONET_Parsed', 'Lightcast_Raw_All', 'Lightcast_Source_Econ', 'Lightcast_Source_Edu', 'AI_Economic', 'Score_Summary', 'Source_Type_By_Source', 'Target_Not_In_AI', 'Lightcast_Parse_Errors', 'SOC_Title_Lookup', 'Crosswalk_Used', 'Recommended_Direct_Only', 'Recommended_Gateway_Top3', 'Direct_Source_Summary_v2', 'Gateway_Source_Summary_v2']

SHEET: Related_Occs_Final
Shape: (1128, 19)

Columns:
['Source_SOC', 'Source_Occupation', 'Source_AI_Exposure_Group', 'Transition_Type', 'Gateway_SOC', 'Gateway_Occupation', 'Gateway_AI_Exposure_Group', 'Target_SOC', 'Target_Occupation', 'Target_AI_Exposure_Group', 'Destination_SOC', 'Destination_Occupation', 'Destination_AI_Exposure_Group', 'Final_Pathway_Score_100', 'Recommendation_Flag', 'Gateway_Recommendation_Flag', 'Source_Type', 'ONET_Tier', 'Final_Compat_Index']

First 5 rows:


,Source_SOC,Source_Occupation,Source_AI_Exposure_Group,Transition_Type,Gateway_SOC,Gateway_Occupation,Gateway_AI_Exposure_Group,Target_SOC,Target_Occupation,Target_AI_Exposure_Group,Destination_SOC,Destination_Occupation,Destination_AI_Exposure_Group,Final_Pathway_Score_100,Recommendation_Flag,Gateway_Recommendation_Flag,Source_Type,ONET_Tier,Final_Compat_Index
0,11-1021,General and Operations Managers,High,Direct,NaN,NaN,NaN,11-3012,Administrative Services Managers,Medium,11-3012,Administrative Services Managers,Medium,100.0,Recommend,NaN,Both,Primary-Short,95.0
1,11-1021,General and Operations Managers,High,Direct,NaN,NaN,NaN,11-3013,Facilities Managers,Medium,11-3013,Facilities Managers,Medium,52.5,Recommend,NaN,ONET,Primary-Short,NaN
2,11-1021,General and Operations Managers,High,Direct,NaN,NaN,NaN,11-3051,Industrial Production Managers,Medium,11-3051,Industrial Production Managers,Medium,49.1,Recommend,NaN,ONET,Primary-Long,NaN
3,11-2011,Advertising and Promotions Managers,High,Direct,NaN,NaN,NaN,13-1111,Management Analysts,Medium,13-1111,Management Analysts,Medium,77.7,Recommend,NaN,Both,Supplemental,94.0
4,11-2021,Marketing Managers,High,Direct,NaN,NaN,NaN,13-1111,Management Analysts,Medium,13-1111,Management Analysts,Medium,73.1,Recommend,NaN,Both,Supplemental,95.0



SHEET: Final_Transitions
Shape: (2673, 75)

Columns:
['Source_SOC', 'Source_Occupation', 'Source_AI_Exposure_Group', 'Target_SOC', 'Target_Occupation', 'Target_AI_Exposure_Group', 'Source_Type', 'Lightcast_Rank', 'ONET_Rank', 'Both_Average_Source_Rank', 'Both_Combined_Rank', 'Final_Compat_Index', 'Lightcast_Compat_Score', 'Lightcast_Rank_Score', 'Lightcast_Combined_Score', 'ONET_Tier', 'ONET_Index', 'ONET_Tier_Score', 'ONET_Rank_Score', 'ONET_Combined_Score', 'Both_Combined_Score', 'Both_Combined_Score_100', 'Preferred_Median_Annual_Wage', 'Preferred_Annual_Openings', 'Preferred_Annual_Percent_Change', 'Preferred_Growth_Rate', 'Preferred_Education', 'Preferred_Work_Experience', 'AI_Annual_Total_Openings', 'AI_Annual_Percent_Change', 'AI_Education_Value', 'AI_Work_Experience_Value', 'AI_Growth_Rate', 'AI_50th_Percentile_Wage', 'LC_Target_Median_Hourly_Earnings', 'LC_Target_Median_Annual_Earnings', 'LC_Target_2024_Jobs', 'LC_Target_2025_Jobs', 'LC_Target_2024_2025_Change', 'LC_Target_20

,Source_SOC,Source_Occupation,Source_AI_Exposure_Group,Target_SOC,Target_Occupation,Target_AI_Exposure_Group,Source_Type,Lightcast_Rank,ONET_Rank,Both_Average_Source_Rank,...,wage_change_pct,wage_delta_score,wage_score,openings_score,growth_score,generic_penalty,direct_realism_score,direct_realism_score_100,direct_new_rank,Recommendation_Flag
0,11-1011,Chief Executives,High,11-9151,Social and Community Service Managers,Medium,ONET,NaN,3.0,3.0,...,-0.501494,0.00,0.714723,0.647587,0.830153,0.0,0.385544,38.6,1,Review / Weak Match
1,11-1011,Chief Executives,High,11-1031,Legislators,Medium,ONET,NaN,11.0,11.0,...,NaN,0.25,NaN,0.276281,0.467078,0.0,0.364064,36.4,2,Review / Weak Match
2,11-1011,Chief Executives,High,11-3012,Administrative Services Managers,Medium,ONET,NaN,15.0,15.0,...,-0.337273,0.25,0.883792,0.760195,0.411523,0.0,0.335943,33.6,3,Review / Weak Match
3,11-1011,Chief Executives,High,13-1111,Management Analysts,Medium,ONET,NaN,14.0,14.0,...,-0.405538,0.00,0.831804,0.913206,0.563786,0.0,0.232032,23.2,4,Review / Weak Match
4,11-1011,Chief Executives,High,39-1014,First-Line Supervisors of Entertainment and Re...,Medium,ONET,NaN,28.0,28.0,...,-0.693738,0.00,0.286806,0.728208,0.813880,0.0,0.204885,20.5,5,Review / Weak Match



SHEET: Clean_Direct_Lower_Only
Shape: (2673, 75)

Columns:
['Source_SOC', 'Source_Occupation', 'Source_AI_Exposure_Group', 'Target_SOC', 'Target_Occupation', 'Target_AI_Exposure_Group', 'Source_Type', 'Lightcast_Rank', 'ONET_Rank', 'Both_Average_Source_Rank', 'Both_Combined_Rank', 'Final_Compat_Index', 'Lightcast_Compat_Score', 'Lightcast_Rank_Score', 'Lightcast_Combined_Score', 'ONET_Tier', 'ONET_Index', 'ONET_Tier_Score', 'ONET_Rank_Score', 'ONET_Combined_Score', 'Both_Combined_Score', 'Both_Combined_Score_100', 'Preferred_Median_Annual_Wage', 'Preferred_Annual_Openings', 'Preferred_Annual_Percent_Change', 'Preferred_Growth_Rate', 'Preferred_Education', 'Preferred_Work_Experience', 'AI_Annual_Total_Openings', 'AI_Annual_Percent_Change', 'AI_Education_Value', 'AI_Work_Experience_Value', 'AI_Growth_Rate', 'AI_50th_Percentile_Wage', 'LC_Target_Median_Hourly_Earnings', 'LC_Target_Median_Annual_Earnings', 'LC_Target_2024_Jobs', 'LC_Target_2025_Jobs', 'LC_Target_2024_2025_Change', 'LC_Tar

...

,Pathway_Type,Gateway_Pathway_Label,Source_SOC,Source_Occupation,Source_AI_Exposure_Group,Gateway_Rank,Gateway_SOC,Gateway_Occupation,Gateway_AI_Exposure_Group,Source_Type,...,destination_generic_penalty,gateway_realism_score,gateway_realism_score_100,destination_realism_score,destination_realism_score_100,gateway_pathway_score,gateway_pathway_score_100,gateway_pathway_rank,Gateway_Recommendation_Flag,gateway_is_top3_for_source
0,Indirect / Gateway,General and Operations Managers -> Administrat...,11-1021,General and Operations Managers,High,1,11-3012,Administrative Services Managers,Medium,Both,...,0.0,0.87000,87.0,0.750,75.0,0.82200,82.2,1,Recommend,True
1,Indirect / Gateway,General and Operations Managers -> Facilities ...,11-1021,General and Operations Managers,High,2,11-3013,Facilities Managers,Medium,ONET,...,0.0,0.59575,59.6,0.750,75.0,0.65745,65.7,2,Recommend,True
2,Indirect / Gateway,General and Operations Managers -> Facilities ...,11-1021,General and Operations Managers,High,2,11-3013,Facilities Managers,Medium,ONET,...,0.0,0.59575,59.6,0.645,64.5,0.61545,61.5,3,Recommend,True
3,Indirect / Gateway,"Transportation, Storage, and Distribution Mana...",11-3071,"Transportation, Storage, and Distribution Mana...",High,2,11-3012,Administrative Services Managers,Medium,Lightcast,...,0.0,0.76750,76.8,0.750,75.0,0.76050,76.1,1,Recommend,True
4,Indirect / Gateway,"Transportation, Storage, and Distribution Mana...",11-3071,"Transportation, Storage, and Distribution Mana...",High,3,11-3013,Facilities Managers,Medium,ONET,...,0.0,0.58375,58.4,0.750,75.0,0.65025,65.0,2,Recommend,True



SHEET: Direct_Source_Summary_v2
Shape: (429, 6)

Columns:
['Source_SOC', 'Source_Occupation', 'Source_AI_Exposure_Group', 'Direct_Transition_Count', 'Recommended_Direct_Count', 'Max_Direct_Score']

First 5 rows:


,Source_SOC,Source_Occupation,Source_AI_Exposure_Group,Direct_Transition_Count,Recommended_Direct_Count,Max_Direct_Score
0,11-1011,Chief Executives,High,5,0,38.6
1,11-1021,General and Operations Managers,High,8,3,100.0
2,11-2011,Advertising and Promotions Managers,High,3,1,77.7
3,11-2021,Marketing Managers,High,2,1,73.1
4,11-2022,Sales Managers,High,3,0,38.4



SHEET: Gateway_Source_Summary_v2
Shape: (193, 6)

Columns:
['Source_SOC', 'Source_Occupation', 'Source_AI_Exposure_Group', 'Gateway_Pathway_Count', 'Recommended_Gateway_Count', 'Max_Gateway_Pathway_Score']

First 5 rows:


,Source_SOC,Source_Occupation,Source_AI_Exposure_Group,Gateway_Pathway_Count,Recommended_Gateway_Count,Max_Gateway_Pathway_Score
0,11-1021,General and Operations Managers,High,3,3,82.2
1,11-2011,Advertising and Promotions Managers,High,1,0,38.6
2,11-2021,Marketing Managers,High,1,0,43.3
3,11-2032,Public Relations Managers,Very High,1,0,50.1
4,11-3071,"Transportation, Storage, and Distribution Mana...",High,3,3,76.1


In [19]:
import pandas as pd
import numpy as np
import re
from pathlib import Path

# =========================================================
# STEP 4B: CLEAN RELATED OCCUPATION TRANSITION FILE
# =========================================================

related_path = Path(
    r"G:\Shared drives\EO_ECONOMIC ANALYSIS\ea_common\Projects\AI\Related Occupations\High_VeryHigh_AND_Medium_Related_Occs_Final.xlsx"
)

output_folder = Path(
    r"C:\Users\301533\Documents\AI Analysis\Training Programs (ETPL+IPEDS)"
)

output_folder.mkdir(parents=True, exist_ok=True)

output_csv = output_folder / "RELATED_OCCUPATION_TRANSITIONS_CLEAN.csv"
output_excel = output_folder / "RELATED_OCCUPATION_TRANSITIONS_CLEAN.xlsx"

summary_excel = output_folder / "RELATED_OCCUPATION_TRANSITION_SUMMARY.xlsx"

# =========================================================
# LOAD MAIN SHEET
# =========================================================

related = pd.read_excel(
    related_path,
    sheet_name="Related_Occs_Final",
    dtype=str
)

print("Original shape:", related.shape)

# =========================================================
# CLEAN SOC CODES
# =========================================================

def clean_soc(x):

    if pd.isna(x):
        return np.nan

    x = str(x).strip()

    # Fix Excel corruption like Nov-1011
    month_map = {
        "jan": "01",
        "feb": "02",
        "mar": "03",
        "apr": "04",
        "may": "05",
        "jun": "06",
        "jul": "07",
        "aug": "08",
        "sep": "09",
        "oct": "10",
        "nov": "11",
        "dec": "12"
    }

    lower = x.lower()

    for month, prefix in month_map.items():
        if lower.startswith(month + "-"):
            x = prefix + x[3:]

    x = re.sub(r"[^0-9\-]", "", x)

    if len(x) == 7 and "-" in x:
        return x

    digits = re.sub(r"[^0-9]", "", x)

    if len(digits) >= 6:
        return digits[:2] + "-" + digits[2:6]

    return np.nan

# =========================================================
# CLEAN KEY SOC FIELDS
# =========================================================

soc_col_map = {
    "Source_SOC_Clean": ["Source SOC", "Source_SOC"],
    "Gateway_SOC_Clean": ["Gateway SOC", "Gateway_SOC"],
    "Related_Occupation_SOC_Clean": ["Related Occupation SOC", "Target_SOC", "Destination_SOC"]
}

for new_col, possible_cols in soc_col_map.items():
    found_col = None

    for col in possible_cols:
        if col in related.columns:
            found_col = col
            break

    if found_col is None:
        print(f"Missing SOC column for {new_col}. Tried: {possible_cols}")
        related[new_col] = np.nan
    else:
        related[new_col] = related[found_col].apply(clean_soc)
        print(f"Created {new_col} from {found_col}")
# =========================================================
# CREATE FINAL DESTINATION SOC
# This is the occupation we map training to
# =========================================================

related["Destination_SOC"] = related["Related_Occupation_SOC_Clean"]

related["Destination_Occupation"] = related["Target_Occupation"]

# =========================================================
# CLEAN SCORES
# =========================================================

score_cols = [
    "Combined Score 0-100",
    "Combined Lightcast + O*NET Score",
    "Lightcast Score",
    "Related Occupation O*NET Score",
    "Combined Rank",
    "Average Source Rank"
]

for col in score_cols:

    if col in related.columns:

        related[col] = pd.to_numeric(
            related[col],
            errors="coerce"
        )

## =========================================================
# CREATE FLAGS
# =========================================================

related["Pathway Type"] = related["Transition_Type"]

related["is_direct"] = (
    related["Transition_Type"]
    .astype(str)
    .str.lower()
    .eq("direct")
)

related["is_gateway"] = (
    related["Transition_Type"]
    .astype(str)
    .str.lower()
    .str.contains("gateway|indirect|stepping", na=False)
)

# =========================================================
# CREATE TRANSITION UID
# =========================================================

related["transition_uid"] = (
    related["Source_SOC_Clean"].astype(str)
    + "_TO_" +
    related["Destination_SOC"].astype(str)
    + "_" +
    related["Pathway Type"].astype(str)
)

# =========================================================
# REMOVE BAD ROWS
# =========================================================

before = related.shape[0]

related = related[
    related["Source_SOC_Clean"].notna() &
    related["Destination_SOC"].notna()
].copy()

after = related.shape[0]

print("\nRows removed due to missing SOC:", before - after)

# =========================================================
# REMOVE EXACT DUPLICATES
# =========================================================

before_dupes = related.shape[0]

related = related.drop_duplicates(
    subset=[
        "transition_uid"
    ]
)

after_dupes = related.shape[0]

print("Duplicate transitions removed:", before_dupes - after_dupes)

# =========================================================
# SUMMARY TABLES
# =========================================================

summary_pathway = (
    related.groupby("Transition_Type")
    .agg(
        transitions=("transition_uid", "count"),
        source_occupations=("Source_SOC_Clean", "nunique"),
        destination_occupations=("Destination_SOC", "nunique")
    )
    .reset_index()
)

summary_exposure = (
    related.groupby("Source_AI_Exposure_Group")
    .agg(
        transitions=("transition_uid", "count"),
        source_occupations=("Source_SOC_Clean", "nunique")
    )
    .reset_index()
)

summary_relationship = (
    related.groupby("Source_Type")
    .agg(
        transitions=("transition_uid", "count")
    )
    .reset_index()
)

# =========================================================
# SAVE SUMMARY FILE
# =========================================================

with pd.ExcelWriter(summary_excel, engine="openpyxl") as writer:

    summary_pathway.to_excel(
        writer,
        sheet_name="By_Pathway_Type",
        index=False
    )

    summary_exposure.to_excel(
        writer,
        sheet_name="By_AI_Exposure",
        index=False
    )

    summary_relationship.to_excel(
        writer,
        sheet_name="By_Relationship_Source",
        index=False
    )

# =========================================================
# SAVE CLEAN FILE
# =========================================================

related.to_csv(output_csv, index=False)
related.to_excel(output_excel, index=False)

# =========================================================
# FINAL SUMMARY
# =========================================================

print("\nFINAL TRANSITION SUMMARY")
print("=" * 60)

print("Final shape:", related.shape)

print("\nPathway types:")
print(related["Pathway Type"].value_counts(dropna=False))

print("\nRelationship sources:")
print(related["Source_Type"].value_counts(dropna=False))

print("\nUnique source occupations:")
print(related["Source_SOC_Clean"].nunique())

print("\nUnique destination occupations:")
print(related["Destination_SOC"].nunique())

print("\nSaved clean transition file to:")
print(output_csv)
print(output_excel)

print("\nSaved summary workbook to:")
print(summary_excel)

display(related.head(25))

Original shape: (1128, 19)
Created Source_SOC_Clean from Source_SOC
Created Gateway_SOC_Clean from Gateway_SOC
Created Related_Occupation_SOC_Clean from Target_SOC

Rows removed due to missing SOC: 0
Duplicate transitions removed: 16

FINAL TRANSITION SUMMARY
Final shape: (1112, 26)

Pathway types:
Pathway Type
Direct     880
Gateway    232
Name: count, dtype: int64

Relationship sources:
Source_Type
ONET         560
Both         305
Lightcast    247
Name: count, dtype: int64

Unique source occupations:
283

Unique destination occupations:
239

Saved clean transition file to:
C:\Users\301533\Documents\AI Analysis\Training Programs (ETPL+IPEDS)\RELATED_OCCUPATION_TRANSITIONS_CLEAN.csv
C:\Users\301533\Documents\AI Analysis\Training Programs (ETPL+IPEDS)\RELATED_OCCUPATION_TRANSITIONS_CLEAN.xlsx

Saved summary workbook to:
C:\Users\301533\Documents\AI Analysis\Training Programs (ETPL+IPEDS)\RELATED_OCCUPATION_TRANSITION_SUMMARY.xlsx


,Source_SOC,Source_Occupation,Source_AI_Exposure_Group,Transition_Type,Gateway_SOC,Gateway_Occupation,Gateway_AI_Exposure_Group,Target_SOC,Target_Occupation,Target_AI_Exposure_Group,...,Source_Type,ONET_Tier,Final_Compat_Index,Source_SOC_Clean,Gateway_SOC_Clean,Related_Occupation_SOC_Clean,Pathway Type,is_direct,is_gateway,transition_uid
0,11-1021,General and Operations Managers,High,Direct,NaN,NaN,NaN,11-3012,Administrative Services Managers,Medium,...,Both,Primary-Short,95,11-1021,NaN,11-3012,Direct,True,False,11-1021_TO_11-3012_Direct
1,11-1021,General and Operations Managers,High,Direct,NaN,NaN,NaN,11-3013,Facilities Managers,Medium,...,ONET,Primary-Short,NaN,11-1021,NaN,11-3013,Direct,True,False,11-1021_TO_11-3013_Direct
2,11-1021,General and Operations Managers,High,Direct,NaN,NaN,NaN,11-3051,Industrial Production Managers,Medium,...,ONET,Primary-Long,NaN,11-1021,NaN,11-3051,Direct,True,False,11-1021_TO_11-3051_Direct
3,11-2011,Advertising and Promotions Managers,High,Direct,NaN,NaN,NaN,13-1111,Management Analysts,Medium,...,Both,Supplemental,94,11-2011,NaN,13-1111,Direct,True,False,11-2011_TO_13-1111_Direct
4,11-2021,Marketing Managers,High,Direct,NaN,NaN,NaN,13-1111,Management Analysts,Medium,...,Both,Supplemental,95,11-2021,NaN,13-1111,Direct,True,False,11-2021_TO_13-1111_Direct
5,11-2032,Public Relations Managers,Very High,Direct,NaN,NaN,NaN,11-3012,Administrative Services Managers,Medium,...,ONET,Supplemental,NaN,11-2032,NaN,11-3012,Direct,True,False,11-2032_TO_11-3012_Direct
6,11-2033,Fundraising Managers,Very High,Direct,NaN,NaN,NaN,11-9151,Social and Community Service Managers,Medium,...,ONET,Primary-Short,NaN,11-2033,NaN,11-9151,Direct,True,False,11-2033_TO_11-9151_Direct
7,11-3061,Purchasing Managers,Very High,Direct,NaN,NaN,NaN,11-3051,Industrial Production Managers,Medium,...,ONET,Supplemental,NaN,11-3061,NaN,11-3051,Direct,True,False,11-3061_TO_11-3051_Direct
8,11-3071,"Transportation, Storage, and Distribution Mana...",High,Direct,NaN,NaN,NaN,11-3012,Administrative Services Managers,Medium,...,Lightcast,NaN,95,11-3071,NaN,11-3012,Direct,True,False,11-3071_TO_11-3012_Direct
9,11-3071,"Transportation, Storage, and Distribution Mana...",High,Direct,NaN,NaN,NaN,11-3051,Industrial Production Managers,Medium,...,ONET,Primary-Short,NaN,11-3071,NaN,11-3051,Direct,True,False,11-3071_TO_11-3051_Direct


In [15]:
print(related.columns.tolist())

['Source_SOC', 'Source_Occupation', 'Source_AI_Exposure_Group', 'Transition_Type', 'Gateway_SOC', 'Gateway_Occupation', 'Gateway_AI_Exposure_Group', 'Target_SOC', 'Target_Occupation', 'Target_AI_Exposure_Group', 'Destination_SOC', 'Destination_Occupation', 'Destination_AI_Exposure_Group', 'Final_Pathway_Score_100', 'Recommendation_Flag', 'Gateway_Recommendation_Flag', 'Source_Type', 'ONET_Tier', 'Final_Compat_Index', 'Source_SOC_Clean', 'Gateway_SOC_Clean', 'Related_Occupation_SOC_Clean']


In [20]:
import pandas as pd
from pathlib import Path

# =========================================================
# STEP 5A: INSPECT SOC-CIP CROSSWALK
# =========================================================

crosswalk_path = Path(
    r"C:\Users\301533\Downloads\CIP2020_SOC2018_Crosswalk (2).xlsx"
)

output_folder = Path(
    r"C:\Users\301533\Documents\AI Analysis\Training Programs (ETPL+IPEDS)"
)

output_folder.mkdir(parents=True, exist_ok=True)

inspection_output = output_folder / "SOC_CIP_CROSSWALK_INSPECTION_step5A.xlsx"

# =========================================================
# INSPECT SHEETS
# =========================================================

xls = pd.ExcelFile(crosswalk_path)

print("Sheet names:")
print(xls.sheet_names)

for sheet in xls.sheet_names:
    print("\n" + "=" * 80)
    print("SHEET:", sheet)
    print("=" * 80)

    df = pd.read_excel(crosswalk_path, sheet_name=sheet, dtype=str)

    print("Shape:", df.shape)
    print("\nColumns:")
    print(df.columns.tolist())

    print("\nFirst 5 rows:")
    display(df.head())

# =========================================================
# SAVE PREVIEWS
# =========================================================

with pd.ExcelWriter(inspection_output, engine="openpyxl") as writer:
    for sheet in xls.sheet_names:
        df = pd.read_excel(crosswalk_path, sheet_name=sheet, dtype=str)
        safe_sheet = sheet[:31]
        df.head(100).to_excel(writer, sheet_name=safe_sheet, index=False)

print("\nSaved inspection workbook to:")
print(inspection_output)

Sheet names:
['File Guide', 'CIP-SOC', 'SOC-CIP', 'New CIP', 'New SOC', 'Added Matches', 'Unmatched CIP Codes', 'Unmatched SOC Codes']

SHEET: File Guide
Shape: (7, 2)

Columns:
['File Name', 'Description']

First 5 rows:


,File Name,Description
0,CIP-SOC,This file crosswalks 2020 CIP Codes to 2018 SO...
1,SOC-CIP,This file crosswalks 2018 SOC Codes to 2020 CI...
2,New CIP,This file contains only NEW 2020 CIP Codes. It...
3,New SOC,This file contains only NEW 2018 SOC Codes. It...
4,Added Matches,This file contains only NEW matches that were ...



SHEET: CIP-SOC
Shape: (6097, 4)

Columns:
['CIP2020Code', 'CIP2020Title', 'SOC2018Code', 'SOC2018Title']

First 5 rows:


,CIP2020Code,CIP2020Title,SOC2018Code,SOC2018Title
0,01.0000,"Agriculture, General.",19-1011,Animal Scientists
1,01.0000,"Agriculture, General.",19-1012,Food Scientists and Technologists
2,01.0000,"Agriculture, General.",19-1013,Soil and Plant Scientists
3,01.0000,"Agriculture, General.",19-4012,Agricultural Technicians
4,01.0000,"Agriculture, General.",25-1041,"Agricultural Sciences Teachers, Postsecondary"



SHEET: SOC-CIP
Shape: (6093, 4)

Columns:
['SOC2018Code', 'SOC2018Title', 'CIP2020Code', 'CIP2020Title']

First 5 rows:


...


SHEET: Unmatched CIP Codes
Shape: (194, 4)

Columns:
['CIP2020Code', 'CIP2020Title', 'SOC2018Code', 'SOC2018Title']

First 5 rows:


,CIP2020Code,CIP2020Title,SOC2018Code,SOC2018Title
0,01.0508,Taxidermy/Taxidermist.,99-9999,NO MATCH
1,01.0599,"Agricultural and Domestic Animal Services, Other.",99-9999,NO MATCH
2,01.0699,Applied Horticulture/Horticultural Business Se...,99-9999,NO MATCH
3,01.0899,"Agricultural Public Services, Other.",99-9999,NO MATCH
4,01.1302,Pre-Veterinary Studies.,99-9999,NO MATCH



SHEET: Unmatched SOC Codes
Shape: (180, 4)

Columns:
['SOC2018Code', 'SOC2018Title', 'CIP2020Code', 'CIP2020Title']

First 5 rows:


,SOC2018Code,SOC2018Title,CIP2020Code,CIP2020Title
0,13-1074,Farm Labor Contractors,99.9999,NO MATCH
1,25-3031,"Substitute Teachers, Short-Term",99.9999,NO MATCH
2,25-3041,Tutors,99.9999,NO MATCH
3,27-1023,Floral Designers,99.9999,NO MATCH
4,27-2023,"Umpires, Referees, and Other Sports Officials",99.9999,NO MATCH



Saved inspection workbook to:
C:\Users\301533\Documents\AI Analysis\Training Programs (ETPL+IPEDS)\SOC_CIP_CROSSWALK_INSPECTION_step5A.xlsx


In [22]:
import pandas as pd
import numpy as np
import re
from pathlib import Path

# =========================================================
# STEP 5B: MERGE TRANSITIONS TO SOC-CIP TO TRAINING PROGRAMS
# =========================================================

folder = Path(r"C:\Users\301533\Documents\AI Analysis\Training Programs (ETPL+IPEDS)")

transitions_path = folder / "RELATED_OCCUPATION_TRANSITIONS_CLEAN.xlsx"
training_path = folder / "TRAINING_PROGRAMS_MASTER_FINAL.xlsx"

crosswalk_path = Path(
    r"C:\Users\301533\Downloads\CIP2020_SOC2018_Crosswalk (2).xlsx"
)

output_csv = folder / "TRANSITION_TRAINING_PROGRAMS_MASTER.csv"
output_excel = folder / "TRANSITION_TRAINING_PROGRAMS_MASTER.xlsx"
summary_excel = folder / "TRANSITION_TRAINING_PROGRAMS_SUMMARY.xlsx"

# =========================================================
# LOAD DATA
# =========================================================

transitions = pd.read_excel(transitions_path, dtype=str)
training = pd.read_excel(training_path, dtype=str)

soc_cip = pd.read_excel(
    crosswalk_path,
    sheet_name="SOC-CIP",
    dtype=str
)

print("Transitions shape:", transitions.shape)
print("Training programs shape:", training.shape)
print("SOC-CIP crosswalk shape:", soc_cip.shape)

# =========================================================
# CLEAN SOC AND CIP CODES
# =========================================================

def clean_soc(x):
    if pd.isna(x):
        return np.nan
    
    x = str(x).strip()
    x = re.sub(r"[^0-9\-]", "", x)
    
    digits = re.sub(r"[^0-9]", "", x)
    
    if len(digits) >= 6:
        return digits[:2] + "-" + digits[2:6]
    
    return np.nan


def clean_cip(x):
    if pd.isna(x):
        return np.nan
    
    x = str(x).strip()
    x = x.replace(".0", "")
    x = re.sub(r"[^0-9]", "", x)
    
    if x == "":
        return np.nan
    
    x = x.zfill(6)
    return x[:2] + "." + x[2:]


transitions["Destination_SOC_Clean"] = transitions["Destination_SOC"].apply(clean_soc)
transitions["Source_SOC_Clean_2"] = transitions["Source_SOC_Clean"].apply(clean_soc)

soc_cip["SOC2018Code_Clean"] = soc_cip["SOC2018Code"].apply(clean_soc)
soc_cip["CIP2020Code_Clean"] = soc_cip["CIP2020Code"].apply(clean_cip)

training["cip_code_clean"] = training["cip_code"].apply(clean_cip)

# =========================================================
# MERGE 1: TRANSITIONS TO SOC-CIP CROSSWALK
# Destination SOC -> CIP
# =========================================================

transition_cip = transitions.merge(
    soc_cip,
    left_on="Destination_SOC_Clean",
    right_on="SOC2018Code_Clean",
    how="left",
    indicator="soc_cip_merge_status"
)

print("\nAfter transition -> SOC-CIP merge:", transition_cip.shape)
print(transition_cip["soc_cip_merge_status"].value_counts(dropna=False))

# =========================================================
# MERGE 2: TRANSITION-CIP TO TRAINING PROGRAMS
# CIP -> ETPL/IPEDS programs
# =========================================================

transition_training = transition_cip.merge(
    training,
    left_on="CIP2020Code_Clean",
    right_on="cip_code_clean",
    how="left",
    suffixes=("", "_training"),
    indicator="training_merge_status"
)

print("\nAfter CIP -> training merge:", transition_training.shape)
print(transition_training["training_merge_status"].value_counts(dropna=False))

# =========================================================
# CREATE ANALYSIS FLAGS
# =========================================================

transition_training["has_cip_match"] = transition_training["CIP2020Code_Clean"].notna()
transition_training["has_training_program"] = transition_training["program_name"].notna()

transition_training["has_etpl_program"] = (
    transition_training["training_source"]
    .astype(str)
    .eq("ETPL")
)

transition_training["has_ipeds_program"] = (
    transition_training["training_source"]
    .astype(str)
    .eq("IPEDS")
)

transition_training["is_wioa_approved_program"] = (
    transition_training["WIOA_approved"]
    .astype(str)
    .str.lower()
    .eq("yes")
)

transition_training["is_short_term_training"] = (
    pd.to_numeric(
        transition_training["duration_weeks"],
        errors="coerce"
    ) <= 26
)

transition_training["transition_training_uid"] = (
    transition_training["transition_uid"].astype(str)
    + "_CIP_" +
    transition_training["CIP2020Code_Clean"].astype(str)
    + "_PROGRAM_" +
    transition_training["program_uid"].astype(str)
)

# =========================================================
# CREATE OCCUPATION-LEVEL SUMMARY
# One row per Source -> Destination transition
# =========================================================

transition_summary = (
    transition_training
        .groupby([
        "transition_uid",
        "Transition_Type",
        "Source_SOC_Clean",
        "Source_Occupation",
        "Source_AI_Exposure_Group",
        "Destination_SOC",
        "Destination_Occupation",
        "Source_Type"
    ], dropna=False)
    .agg(
        related_cip_count=("CIP2020Code_Clean", "nunique"),
        training_program_count=("program_uid", "nunique"),
        etpl_program_count=("program_uid", lambda x: x[transition_training.loc[x.index, "training_source"].eq("ETPL")].nunique()),
        ipeds_program_count=("program_uid", lambda x: x[transition_training.loc[x.index, "training_source"].eq("IPEDS")].nunique()),
        wioa_program_count=("program_uid", lambda x: x[transition_training.loc[x.index, "is_wioa_approved_program"]].nunique()),
        online_program_count=("program_uid", lambda x: x[transition_training.loc[x.index, "is_online"].astype(str).str.lower().eq("true")].nunique()),
        short_term_program_count=("program_uid", lambda x: x[transition_training.loc[x.index, "is_short_term_training"]].nunique()),
        avg_total_instate_cost=("total_instate_cost", lambda x: pd.to_numeric(x, errors="coerce").mean()),
        median_total_instate_cost=("total_instate_cost", lambda x: pd.to_numeric(x, errors="coerce").median()),
        avg_duration_weeks=("duration_weeks", lambda x: pd.to_numeric(x, errors="coerce").mean())
    )
    .reset_index()
)

transition_summary["has_any_training_program"] = (
    transition_summary["training_program_count"] > 0
)

transition_summary["has_any_etpl_program"] = (
    transition_summary["etpl_program_count"] > 0
)

transition_summary["has_any_wioa_program"] = (
    transition_summary["wioa_program_count"] > 0
)

# =========================================================
# CREATE DESTINATION OCCUPATION SUMMARY
# One row per lower-exposure destination occupation
# =========================================================

destination_summary = (
    transition_training
    .groupby([
        "Destination_SOC",
        "Destination_Occupation"
    ], dropna=False)
    .agg(
        source_occupation_count=("Source_SOC_Clean", "nunique"),
        transition_count=("transition_uid", "nunique"),
        related_cip_count=("CIP2020Code_Clean", "nunique"),
        training_program_count=("program_uid", "nunique"),
        etpl_program_count=("program_uid", lambda x: x[transition_training.loc[x.index, "training_source"].eq("ETPL")].nunique()),
        ipeds_program_count=("program_uid", lambda x: x[transition_training.loc[x.index, "training_source"].eq("IPEDS")].nunique()),
        wioa_program_count=("program_uid", lambda x: x[transition_training.loc[x.index, "is_wioa_approved_program"]].nunique()),
        avg_total_instate_cost=("total_instate_cost", lambda x: pd.to_numeric(x, errors="coerce").mean()),
        median_total_instate_cost=("total_instate_cost", lambda x: pd.to_numeric(x, errors="coerce").median()),
        avg_duration_weeks=("duration_weeks", lambda x: pd.to_numeric(x, errors="coerce").mean())
    )
    .reset_index()
)

destination_summary["has_any_training_program"] = (
    destination_summary["training_program_count"] > 0
)

destination_summary["has_any_etpl_program"] = (
    destination_summary["etpl_program_count"] > 0
)

destination_summary["has_any_wioa_program"] = (
    destination_summary["wioa_program_count"] > 0
)

# =========================================================
# CREATE CIP-LEVEL SUMMARY
# =========================================================

cip_summary = (
    transition_training
    .groupby([
        "CIP2020Code_Clean",
        "CIP2020Title"
    ], dropna=False)
    .agg(
        destination_occupation_count=("Destination_SOC", "nunique"),
        transition_count=("transition_uid", "nunique"),
        training_program_count=("program_uid", "nunique"),
        etpl_program_count=("program_uid", lambda x: x[transition_training.loc[x.index, "training_source"].eq("ETPL")].nunique()),
        ipeds_program_count=("program_uid", lambda x: x[transition_training.loc[x.index, "training_source"].eq("IPEDS")].nunique()),
        wioa_program_count=("program_uid", lambda x: x[transition_training.loc[x.index, "is_wioa_approved_program"]].nunique())
    )
    .reset_index()
)

# =========================================================
# SAVE OUTPUTS
# =========================================================

transition_training.to_csv(output_csv, index=False)
transition_training.to_excel(output_excel, index=False)

with pd.ExcelWriter(summary_excel, engine="openpyxl") as writer:
    transition_summary.to_excel(writer, sheet_name="Transition_Summary", index=False)
    destination_summary.to_excel(writer, sheet_name="Destination_Summary", index=False)
    cip_summary.to_excel(writer, sheet_name="CIP_Summary", index=False)

# =========================================================
# FINAL PRINT SUMMARY
# =========================================================

print("\nFINAL MERGED DATASET SUMMARY")
print("=" * 60)

print("Final program-level shape:", transition_training.shape)

print("\nSOC-CIP merge status:")
print(transition_training["soc_cip_merge_status"].value_counts(dropna=False))

print("\nTraining merge status:")
print(transition_training["training_merge_status"].value_counts(dropna=False))

print("\nUnique transitions:")
print(transition_training["transition_uid"].nunique())

print("\nUnique destination occupations:")
print(transition_training["Destination_SOC"].nunique())

print("\nUnique CIPs matched:")
print(transition_training["CIP2020Code_Clean"].nunique())

print("\nUnique training programs matched:")
print(transition_training["program_uid"].nunique())

print("\nTransitions with at least one training program:")
print(transition_summary["has_any_training_program"].sum())

print("\nDestination occupations with at least one training program:")
print(destination_summary["has_any_training_program"].sum())

print("\nSaved files:")
print(output_csv)
print(output_excel)
print(summary_excel)

display(transition_training.head(25))

Transitions shape: (1112, 26)
Training programs shape: (3828, 43)
SOC-CIP crosswalk shape: (6093, 4)

After transition -> SOC-CIP merge: (9662, 35)
soc_cip_merge_status
both          9662
left_only        0
right_only       0
Name: count, dtype: int64

After CIP -> training merge: (36394, 80)
training_merge_status
both          31610
left_only      4784
right_only        0
Name: count, dtype: int64


c:\Users\301533\AppData\Local\Programs\Python\Python313\Lib\site-packages\numpy\lib\_nanfunctions_impl.py:1214: RuntimeWarning: Mean of empty slice
  return np.nanmean(a, axis, out=out, keepdims=keepdims)
c:\Users\301533\AppData\Local\Programs\Python\Python313\Lib\site-packages\numpy\lib\_nanfunctions_impl.py:1214: RuntimeWarning: Mean of empty slice
  return np.nanmean(a, axis, out=out, keepdims=keepdims)
c:\Users\301533\AppData\Local\Programs\Python\Python313\Lib\site-packages\numpy\lib\_nanfunctions_impl.py:1214: RuntimeWarning: Mean of empty slice
  return np.nanmean(a, axis, out=out, keepdims=keepdims)
c:\Users\301533\AppData\Local\Programs\Python\Python313\Lib\site-packages\numpy\lib\_nanfunctions_impl.py:1214: RuntimeWarning: Mean of empty slice
  return np.nanmean(a, axis, out=out, keepdims=keepdims)
c:\Users\301533\AppData\Local\Programs\Python\Python313\Lib\site-packages\numpy\lib\_nanfunctions_impl.py:1214: RuntimeWarning: Mean of empty slice
  return np.nanmean(a, axis, out


FINAL MERGED DATASET SUMMARY
Final program-level shape: (36394, 87)

SOC-CIP merge status:
soc_cip_merge_status
both          36394
left_only         0
right_only        0
Name: count, dtype: int64

Training merge status:
training_merge_status
both          31610
left_only      4784
right_only        0
Name: count, dtype: int64

Unique transitions:
1112

Unique destination occupations:
239

Unique CIPs matched:
761

Unique training programs matched:
1848

Transitions with at least one training program:
863

Destination occupations with at least one training program:
163

Saved files:
C:\Users\301533\Documents\AI Analysis\Training Programs (ETPL+IPEDS)\TRANSITION_TRAINING_PROGRAMS_MASTER.csv
C:\Users\301533\Documents\AI Analysis\Training Programs (ETPL+IPEDS)\TRANSITION_TRAINING_PROGRAMS_MASTER.xlsx
C:\Users\301533\Documents\AI Analysis\Training Programs (ETPL+IPEDS)\TRANSITION_TRAINING_PROGRAMS_SUMMARY.xlsx


,Source_SOC,Source_Occupation,Source_AI_Exposure_Group,Transition_Type,Gateway_SOC,Gateway_Occupation,Gateway_AI_Exposure_Group,Target_SOC,Target_Occupation,Target_AI_Exposure_Group,...,is_in_person,cip_code_clean,training_merge_status,has_cip_match,has_training_program,has_etpl_program,has_ipeds_program,is_wioa_approved_program,is_short_term_training,transition_training_uid
0,11-1021,General and Operations Managers,High,Direct,NaN,NaN,NaN,11-3012,Administrative Services Managers,Medium,...,NaN,NaN,left_only,True,False,False,False,False,False,11-1021_TO_11-3012_Direct_CIP_01.8202_PROGRAM_nan
1,11-1021,General and Operations Managers,High,Direct,NaN,NaN,NaN,11-3012,Administrative Services Managers,Medium,...,True,05.1711,both,True,True,True,False,True,False,11-1021_TO_11-3012_Direct_CIP_05.1711_PROGRAM_...
2,11-1021,General and Operations Managers,High,Direct,NaN,NaN,NaN,11-3012,Administrative Services Managers,Medium,...,False,05.1711,both,True,True,True,False,True,False,11-1021_TO_11-3012_Direct_CIP_05.1711_PROGRAM_...
3,11-1021,General and Operations Managers,High,Direct,NaN,NaN,NaN,11-3012,Administrative Services Managers,Medium,...,False,05.1711,both,True,True,True,False,True,True,11-1021_TO_11-3012_Direct_CIP_05.1711_PROGRAM_...
4,11-1021,General and Operations Managers,High,Direct,NaN,NaN,NaN,11-3012,Administrative Services Managers,Medium,...,False,05.1711,both,True,True,False,True,False,False,11-1021_TO_11-3012_Direct_CIP_05.1711_PROGRAM_...
5,11-1021,General and Operations Managers,High,Direct,NaN,NaN,NaN,11-3012,Administrative Services Managers,Medium,...,False,05.1711,both,True,True,False,True,False,False,11-1021_TO_11-3012_Direct_CIP_05.1711_PROGRAM_...
6,11-1021,General and Operations Managers,High,Direct,NaN,NaN,NaN,11-3012,Administrative Services Managers,Medium,...,False,05.1711,both,True,True,False,True,False,False,11-1021_TO_11-3012_Direct_CIP_05.1711_PROGRAM_...
7,11-1021,General and Operations Managers,High,Direct,NaN,NaN,NaN,11-3012,Administrative Services Managers,Medium,...,False,05.2101,both,True,True,True,False,True,True,11-1021_TO_11-3012_Direct_CIP_05.2101_PROGRAM_...
8,11-1021,General and Operations Managers,High,Direct,NaN,NaN,NaN,11-3012,Administrative Services Managers,Medium,...,False,05.2101,both,True,True,True,False,True,False,11-1021_TO_11-3012_Direct_CIP_05.2101_PROGRAM_...
9,11-1021,General and Operations Managers,High,Direct,NaN,NaN,NaN,11-3012,Administrative Services Managers,Medium,...,False,05.2101,both,True,True,True,False,True,False,11-1021_TO_11-3012_Direct_CIP_05.2101_PROGRAM_...


In [24]:
import pandas as pd
import numpy as np
from pathlib import Path

# =========================================================
# STEP 6: CREATE MANUSCRIPT ANALYTICAL OUTPUTS
# =========================================================

folder = Path(r"C:\Users\301533\Documents\AI Analysis\Training Programs (ETPL+IPEDS)")

master_path = folder / "TRANSITION_TRAINING_PROGRAMS_MASTER.xlsx"
summary_path = folder / "TRANSITION_TRAINING_PROGRAMS_SUMMARY.xlsx"

output_excel = folder / "AI_TRAINING_PATHWAYS_MANUSCRIPT_OUTPUTS.xlsx"

# =========================================================
# LOAD DATA
# =========================================================

master = pd.read_excel(master_path)
transition_summary = pd.read_excel(summary_path, sheet_name="Transition_Summary")
destination_summary = pd.read_excel(summary_path, sheet_name="Destination_Summary")
cip_summary = pd.read_excel(summary_path, sheet_name="CIP_Summary")

print("Master shape:", master.shape)
print("Transition summary shape:", transition_summary.shape)
print("Destination summary shape:", destination_summary.shape)
print("CIP summary shape:", cip_summary.shape)

# =========================================================
# CLEAN NUMERIC FIELDS
# =========================================================

for df in [transition_summary, destination_summary, cip_summary]:
    for col in df.columns:
        if any(x in col.lower() for x in ["count", "cost", "duration", "score", "rank"]):
            df[col] = pd.to_numeric(df[col], errors="ignore")

# =========================================================
# 1. CORE TRANSITION COVERAGE TABLE
# =========================================================

core_transition_table = transition_summary.copy()

core_transition_table["training_coverage_category"] = np.select(
    [
        core_transition_table["training_program_count"] == 0,
        core_transition_table["wioa_program_count"] > 0,
        core_transition_table["etpl_program_count"] > 0,
        core_transition_table["training_program_count"] > 0,
    ],
    [
        "No Arizona training program identified",
        "Has WIOA-approved training option",
        "Has ETPL training option",
        "Has IPEDS-only training option",
    ],
    default="Unknown"
)

core_transition_table["short_term_available"] = (
    core_transition_table["short_term_program_count"] > 0
)

core_transition_table["online_available"] = (
    core_transition_table["online_program_count"] > 0
)

# =========================================================
# 2. DESTINATION OCCUPATION INFRASTRUCTURE RANKING
# =========================================================

destination_rank = destination_summary.copy()

destination_rank["training_access_score"] = (
    destination_rank["training_program_count"].rank(pct=True) * 0.35 +
    destination_rank["etpl_program_count"].rank(pct=True) * 0.25 +
    destination_rank["wioa_program_count"].rank(pct=True) * 0.25 +
    destination_rank["related_cip_count"].rank(pct=True) * 0.15
)

destination_rank = destination_rank.sort_values(
    by="training_access_score",
    ascending=False
)

# =========================================================
# 3. TRANSITION GAP ANALYSIS
# High transition relevance but low/no training availability
# =========================================================

gap_analysis = core_transition_table.copy()

gap_analysis["training_gap_flag"] = (
    gap_analysis["training_program_count"] == 0
)

gap_analysis["limited_training_flag"] = (
    (gap_analysis["training_program_count"] > 0) &
    (gap_analysis["etpl_program_count"] == 0) &
    (gap_analysis["wioa_program_count"] == 0)
)

gap_analysis = gap_analysis[
    gap_analysis["training_gap_flag"] |
    gap_analysis["limited_training_flag"]
].copy()

gap_analysis = gap_analysis.sort_values(
    by=[
        "training_gap_flag",
        "limited_training_flag",
        "Transition_Type",
        "Source_Occupation",
        "Destination_Occupation"
    ],
    ascending=[False, False, True, True, True]
)

# =========================================================
# 4. DIRECT VS GATEWAY COMPARISON
# =========================================================

direct_gateway_comparison = (
    core_transition_table
    .groupby("Transition_Type", dropna=False)
    .agg(
        transitions=("transition_uid", "count"),
        source_occupations=("Source_SOC_Clean", "nunique"),
        destination_occupations=("Destination_SOC", "nunique"),
        avg_training_programs=("training_program_count", "mean"),
        median_training_programs=("training_program_count", "median"),
        transitions_with_training=("has_any_training_program", "sum"),
        transitions_with_etpl=("has_any_etpl_program", "sum"),
        transitions_with_wioa=("has_any_wioa_program", "sum"),
        avg_cost=("avg_total_instate_cost", "mean"),
        median_cost=("median_total_instate_cost", "median"),
        avg_duration_weeks=("avg_duration_weeks", "mean")
    )
    .reset_index()
)

direct_gateway_comparison["pct_with_training"] = (
    direct_gateway_comparison["transitions_with_training"] /
    direct_gateway_comparison["transitions"]
)

direct_gateway_comparison["pct_with_etpl"] = (
    direct_gateway_comparison["transitions_with_etpl"] /
    direct_gateway_comparison["transitions"]
)

direct_gateway_comparison["pct_with_wioa"] = (
    direct_gateway_comparison["transitions_with_wioa"] /
    direct_gateway_comparison["transitions"]
)

# =========================================================
# 5. CIP CLUSTER ANALYSIS
# =========================================================

cip_cluster = cip_summary.copy()

cip_cluster["cip_2digit"] = cip_cluster["CIP2020Code_Clean"].astype(str).str[:2]

cip2_summary = (
    cip_cluster
    .groupby("cip_2digit", dropna=False)
    .agg(
        cip_count=("CIP2020Code_Clean", "nunique"),
        destination_occupation_count=("destination_occupation_count", "sum"),
        transition_count=("transition_count", "sum"),
        training_program_count=("training_program_count", "sum"),
        etpl_program_count=("etpl_program_count", "sum"),
        ipeds_program_count=("ipeds_program_count", "sum"),
        wioa_program_count=("wioa_program_count", "sum")
    )
    .reset_index()
    .sort_values(by="training_program_count", ascending=False)
)

# Optional CIP2 labels
cip2_labels = {
    "01": "Agriculture",
    "03": "Natural Resources",
    "04": "Architecture",
    "05": "Area/Ethnic/Cultural Studies",
    "09": "Communication",
    "10": "Communications Technologies",
    "11": "Computer and Information Sciences",
    "12": "Personal and Culinary Services",
    "13": "Education",
    "14": "Engineering",
    "15": "Engineering Technologies",
    "19": "Family and Consumer Sciences",
    "22": "Legal Professions",
    "24": "Liberal Arts",
    "30": "Multi/Interdisciplinary Studies",
    "31": "Parks/Recreation/Fitness",
    "43": "Homeland Security/Law Enforcement",
    "44": "Public Administration/Social Service",
    "46": "Construction Trades",
    "47": "Mechanic and Repair Technologies",
    "48": "Precision Production",
    "49": "Transportation",
    "50": "Visual and Performing Arts",
    "51": "Health Professions",
    "52": "Business/Management/Marketing"
}

cip2_summary["cip_2digit_title"] = cip2_summary["cip_2digit"].map(cip2_labels)

# =========================================================
# 6. TOP DESTINATIONS WITH TRAINING OPTIONS
# =========================================================

top_destinations = destination_rank[
    destination_rank["training_program_count"] > 0
].head(50).copy()

# =========================================================
# 7. PROGRAM DRILLDOWN SAMPLE FOR DASHBOARD
# =========================================================

program_drilldown = master[
    master["has_training_program"] == True
].copy()

keep_cols = [
    "Pathway Type",
    "Source SOC",
    "Source Occupation",
    "Source AI Exposure",
    "Gateway SOC",
    "Gateway Occupation",
    "Destination_SOC",
    "Destination_Occupation",
    "Relationship Source",
    "Combined Score 0-100",
    "CIP2020Code_Clean",
    "CIP2020Title",
    "training_source",
    "program_name",
    "provider_name",
    "credential_category",
    "credential_name",
    "delivery_mode",
    "duration_weeks",
    "WIOA_approved",
    "total_instate_cost",
    "program_url",
    "has_training_program",
    "training_program_count",
    "training_url"
]

program_drilldown = program_drilldown[
    [c for c in keep_cols if c in program_drilldown.columns]
].copy()

# =========================================================
# SAVE OUTPUT WORKBOOK
# =========================================================

with pd.ExcelWriter(output_excel, engine="openpyxl") as writer:
    core_transition_table.to_excel(writer, sheet_name="1_Core_Transition_Table", index=False)
    destination_rank.to_excel(writer, sheet_name="2_Destination_Ranking", index=False)
    gap_analysis.to_excel(writer, sheet_name="3_Gap_Analysis", index=False)
    direct_gateway_comparison.to_excel(writer, sheet_name="4_Direct_vs_Gateway", index=False)
    cip2_summary.to_excel(writer, sheet_name="5_CIP2_Cluster_Summary", index=False)
    top_destinations.to_excel(writer, sheet_name="6_Top_Destinations", index=False)
    program_drilldown.to_excel(writer, sheet_name="7_Program_Drilldown", index=False)

print("Saved manuscript outputs to:")
print(output_excel)

print("\nKey outputs:")
print("Core transitions:", core_transition_table.shape)
print("Destination ranking:", destination_rank.shape)
print("Gap analysis:", gap_analysis.shape)
print("Direct vs Gateway:", direct_gateway_comparison.shape)
print("CIP2 summary:", cip2_summary.shape)
print("Program drilldown:", program_drilldown.shape)

display(core_transition_table.head())

Master shape: (36394, 87)
Transition summary shape: (1112, 21)
Destination summary shape: (239, 15)
CIP summary shape: (766, 8)


C:\Users\301533\AppData\Local\Temp\ipykernel_19540\1915282878.py:37: FutureWarning: errors='ignore' is deprecated and will raise in a future version. Use to_numeric without passing `errors` and catch exceptions explicitly instead
  df[col] = pd.to_numeric(df[col], errors="ignore")


Saved manuscript outputs to:
C:\Users\301533\Documents\AI Analysis\Training Programs (ETPL+IPEDS)\AI_TRAINING_PATHWAYS_MANUSCRIPT_OUTPUTS.xlsx

Key outputs:
Core transitions: (1112, 24)
Destination ranking: (239, 16)
Gap analysis: (486, 26)
Direct vs Gateway: (2, 15)
CIP2 summary: (22, 9)
Program drilldown: (31610, 17)


,transition_uid,Transition_Type,Source_SOC_Clean,Source_Occupation,Source_AI_Exposure_Group,Destination_SOC,Destination_Occupation,Source_Type,related_cip_count,training_program_count,...,short_term_program_count,avg_total_instate_cost,median_total_instate_cost,avg_duration_weeks,has_any_training_program,has_any_etpl_program,has_any_wioa_program,training_coverage_category,short_term_available,online_available
0,11-1021_TO_11-3012_Direct,Direct,11-1021,General and Operations Managers,High,11-3012,Administrative Services Managers,Both,7,74,...,5,7986.083125,4500.34,32.7,True,True,True,Has WIOA-approved training option,True,True
1,11-1021_TO_11-3012_Gateway,Gateway,11-1021,General and Operations Managers,High,11-3012,Administrative Services Managers,ONET,7,74,...,5,7986.083125,4500.34,32.7,True,True,True,Has WIOA-approved training option,True,True
2,11-1021_TO_11-3013_Direct,Direct,11-1021,General and Operations Managers,High,11-3013,Facilities Managers,ONET,13,87,...,5,5447.487368,3700.00,34.0,True,True,True,Has WIOA-approved training option,True,False
3,11-1021_TO_11-3013_Gateway,Gateway,11-1021,General and Operations Managers,High,11-3013,Facilities Managers,Both,13,87,...,5,5447.487368,3700.00,34.0,True,True,True,Has WIOA-approved training option,True,False
4,11-1021_TO_11-3051_Direct,Direct,11-1021,General and Operations Managers,High,11-3051,Industrial Production Managers,ONET,9,92,...,7,5008.618696,3700.00,32.0,True,True,True,Has WIOA-approved training option,True,False


# Interactive Tool

In [ ]:
# import streamlit as st
# import pandas as pd
# from pathlib import Path

# # =========================================================
# # AI TRAINING PATHWAYS LOOKUP TOOL
# # =========================================================

# st.set_page_config(
#     page_title="AI Transition Training Pathways",
#     layout="wide"
# )

# file_path = Path(
#     r"C:\Users\301533\Documents\AI Analysis\Training Programs (ETPL+IPEDS)\TRANSITION_TRAINING_PROGRAMS_MASTER.xlsx"
# )

# @st.cache_data
# def load_data(path):
#     df = pd.read_excel(path)
#     return df

# df = load_data(file_path)

# # Keep only rows with actual training programs
# df = df[df["has_training_program"] == True].copy()

# st.title("AI Transition Training Pathways")
# st.write(
#     "Select a lower-exposure related occupation to view Arizona training programs linked through SOC-to-CIP mappings."
# )

# # =========================================================
# # SIDEBAR FILTERS
# # =========================================================

# st.sidebar.header("Filters")

# pathway_options = sorted(df["Pathway Type"].dropna().unique())
# selected_pathways = st.sidebar.multiselect(
#     "Pathway Type",
#     pathway_options,
#     default=pathway_options
# )

# source_exposure_options = sorted(df["Source AI Exposure"].dropna().unique())
# selected_exposure = st.sidebar.multiselect(
#     "Source AI Exposure",
#     source_exposure_options,
#     default=source_exposure_options
# )

# destination_options = sorted(df["Destination_Occupation"].dropna().unique())
# selected_destination = st.sidebar.selectbox(
#     "Lower-Exposure Related Occupation",
#     destination_options
# )

# training_source_options = sorted(df["training_source"].dropna().unique())
# selected_training_sources = st.sidebar.multiselect(
#     "Training Source",
#     training_source_options,
#     default=training_source_options
# )

# credential_options = sorted(df["credential_category"].dropna().unique())
# selected_credentials = st.sidebar.multiselect(
#     "Credential / Degree Type",
#     credential_options,
#     default=credential_options
# )

# wioa_only = st.sidebar.checkbox("WIOA approved only")
# online_only = st.sidebar.checkbox("Online only")

# # =========================================================
# # FILTER DATA
# # =========================================================

# filtered = df[
#     (df["Pathway Type"].isin(selected_pathways)) &
#     (df["Source AI Exposure"].isin(selected_exposure)) &
#     (df["Destination_Occupation"] == selected_destination) &
#     (df["training_source"].isin(selected_training_sources)) &
#     (df["credential_category"].isin(selected_credentials))
# ].copy()

# if wioa_only:
#     filtered = filtered[
#         filtered["WIOA_approved"].astype(str).str.lower().eq("yes")
#     ]

# if online_only:
#     filtered = filtered[
#         filtered["delivery_mode"].astype(str).str.contains("online", case=False, na=False)
#     ]

# # =========================================================
# # SUMMARY METRICS
# # =========================================================

# st.subheader(selected_destination)

# col1, col2, col3, col4 = st.columns(4)

# col1.metric("Training Programs", filtered["program_uid"].nunique())
# col2.metric("Providers", filtered["provider_name"].nunique())
# col3.metric("CIP Codes", filtered["CIP2020Code_Clean"].nunique())
# col4.metric(
#     "WIOA Programs",
#     filtered.loc[
#         filtered["WIOA_approved"].astype(str).str.lower().eq("yes"),
#         "program_uid"
#     ].nunique()
# )

# # =========================================================
# # TRAINING PROGRAM TABLE
# # =========================================================

# display_cols = [
#     "program_name",
#     "provider_name",
#     "training_source",
#     "credential_category",
#     "credential_name",
#     "delivery_mode",
#     "duration_weeks",
#     "total_instate_cost",
#     "training_locations",
#     "WIOA_approved",
#     "CIP2020Code_Clean",
#     "CIP2020Title",
#     "program_url",
#     "training_url"
# ]

# display_cols = [c for c in display_cols if c in filtered.columns]

# programs = filtered[display_cols].drop_duplicates().copy()

# programs = programs.sort_values(
#     by=["training_source", "provider_name", "program_name"]
# )

# st.subheader("Training Programs")

# st.dataframe(
#     programs,
#     use_container_width=True,
#     hide_index=True
# )

# # =========================================================
# # DOWNLOAD BUTTON
# # =========================================================

# csv = programs.to_csv(index=False).encode("utf-8")

# st.download_button(
#     label="Download filtered training programs as CSV",
#     data=csv,
#     file_name=f"{selected_destination}_training_programs.csv",
#     mime="text/csv"
# )

# # =========================================================
# # SOURCE OCCUPATION CONTEXT
# # =========================================================

# st.subheader("Related Source Occupations")

# source_cols = [
#     "Source Occupation",
#     "Source SOC",
#     "Source AI Exposure",
#     "Pathway Type",
#     "Gateway Occupation",
#     "Relationship Source",
#     "Combined Score 0-100"
# ]

# source_cols = [c for c in source_cols if c in filtered.columns]

# source_table = (
#     filtered[source_cols]
#     .drop_duplicates()
#     .sort_values(by=["Pathway Type", "Source Occupation"])
# )

# st.dataframe(
#     source_table,
#     use_container_width=True,
#     hide_index=True
# )

2026-06-03 17:03:17.566 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-06-03 17:03:17.568 No runtime found, using MemoryCacheStorageManager
2026-06-03 17:03:17.735 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-06-03 17:03:17.736 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-06-03 17:03:17.736 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-06-03 17:03:17.736 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-06-03 17:03:17.737 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-06-03 17:03:17.737 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-06-03 17:03:17.738 Thread 'MainThread':

KeyError: 'Source AI Exposure'

In [37]:
import pandas as pd
import json
from pathlib import Path

# =========================================================
# CREATE WEBSITE TRAINING PAGE FROM TRAINING PROGRAM DATA
# Keeps ALL occupations, including those with 0 training programs
# =========================================================

folder = Path(r"C:\Users\301533\Documents\AI Analysis\Training Programs (ETPL+IPEDS)")

input_file = folder / "TRANSITION_TRAINING_PROGRAMS_MASTER.csv"

# Change this to your repo path
output_html = Path(
    r"C:\Users\301533\Documents\GitHub\AI\AI Website\aipathways.github.io\transitions.html"
)

# =========================================================
# LOAD DATA
# =========================================================

df = pd.read_csv(input_file, low_memory=False)

df["has_training_program"] = (
    df["has_training_program"]
    .astype(str)
    .str.lower()
    .isin(["true", "1", "yes"])
)

df["training_program_count"] = df.groupby(
    ["Destination_Occupation", "Destination_SOC"]
)["program_name"].transform(lambda x: x.notna().sum())

# =========================================================
# KEEP ONLY NEEDED COLUMNS
# =========================================================

keep_cols = [
    "Destination_Occupation",
    "Destination_SOC",
    "Transition_Type",
    "has_training_program",
    "training_program_count",
    "training_source",
    "program_name",
    "provider_name",
    "credential_category",
    "credential_name",
    "delivery_mode",
    "duration_weeks",
    "duration_hours",
    "total_instate_cost",
    "training_locations",
    "WIOA_approved",
    "program_url",
    "training_url",
    "CIP2020Code_Clean"
]

df = df[[c for c in keep_cols if c in df.columns]].copy()
df = df.fillna("")
df = df.drop_duplicates()

df["destination_display"] = (
    df["Destination_Occupation"].astype(str)
    + " ("
    + df["Destination_SOC"].astype(str)
    + ")"
)

records = df.to_dict(orient="records")
data_json = json.dumps(records, ensure_ascii=False, allow_nan=False)

# =========================================================
# BUILD HTML
# =========================================================

html = f"""
<!DOCTYPE html>
<html lang="en">
<head>
<meta charset="UTF-8">
<title>AI Transition Training Pathways</title>
<meta name="viewport" content="width=device-width, initial-scale=1.0">
<link rel="icon" type="image/png" href="assets/AZ%20ICON%20LOGO_Solid_Agnostic_Color.png" />
<link rel="stylesheet" href="styles.css" />
<script src="https://ajax.googleapis.com/ajax/libs/jquery/3.5.1/jquery.min.js"></script>
<script src="https://static.az.gov/sliver/sliver.js" type="text/javascript"></script>

<style>

body {{
    margin: 0;
    background: #f5f7fa;
    color: #1f2937;
}}

.filters {{
    display: grid;
    grid-template-columns: repeat(2, 1fr);
    gap: 18px;
    background: white;
    padding: 20px;
    border-radius: 16px;
    box-shadow: 0 2px 10px rgba(0,0,0,0.06);
}}

label {{
    font-weight: bold;
    display: block;
    margin-bottom: 8px;
    font-size: 14px;
}}

.tooltip-icon {{
    color: #1f5fbf;
    font-size: 14px;
    margin-left: 4px;
    cursor: help;
    font-weight: 600;
}}

select, input {{
    width: 100%;
    box-sizing: border-box;
    padding: 10px;
    border-radius: 8px;
    border: 1px solid #d4d9df;
    font-size: 14px;
}}

.summary {{
    display: grid;
    grid-template-columns: repeat(4, 1fr);
    gap: 16px;
    margin-top: 22px;
    margin-bottom: 26px;
}}

.card {{
    background: white;
    border-radius: 16px;
    padding: 12px 16px;
    box-shadow: 0 2px 10px rgba(0,0,0,0.06);
}}

.metric {{
    font-size: 22px;
    font-weight: bold;
    color: #1f5fbf;
    margin-bottom: 2px;
}}

.metric-label {{
    margin-top: 2px;
    color: #667085;
}}

.results-header {{
    display: flex;
    justify-content: space-between;
    align-items: center;
    margin-bottom: 14px;
}}

button {{
    background: #17C3B2;
    color: white;
    border: none;
    padding: 12px 16px;
    border-radius: 8px;
    font-weight: bold;
    cursor: pointer;
}}

table {{
    width: 100%;
    border-collapse: collapse;
    background: white;
    border-radius: 16px;
    overflow: hidden;
    box-shadow: 0 2px 10px rgba(0,0,0,0.06);
}}

th {{
    background: #122033;
    color: white;
    text-align: left;
    padding: 14px 10px;
    font-size: 13px;
}}

td {{
    padding: 12px 10px;
    border-bottom: 1px solid #e7ebf0;
    vertical-align: top;
    font-size: 13px;
}}

tr:hover {{
    background: #f3fbfa;
}}

a {{
    color: #1f5fbf;
    font-weight: bold;
}}

.note {{
    margin-top: 16px;
    font-size: 13px;
    color: #667085;
}}

.no-results {{
    background: white;
    border-radius: 16px;
    padding: 18px;
    color: #667085;
    box-shadow: 0 2px 10px rgba(0,0,0,0.06);
}}

@media (max-width: 1000px) {{
    .filters {{
        grid-template-columns: 1fr;
    }}

    .summary {{
        grid-template-columns: 1fr;
    }}
}}

</style>
</head>

<body>

<header class="site-header">
  <div class="wrap site-header-inner">
    <a class="agency-brand" href="https://oeo.az.gov/" aria-label="Arizona Office of Economic Opportunity">
      <img
        src="assets/AZ_OEO_Tertiary_Black_2000px.png"
        alt="Arizona Office of Economic Opportunity"
      />
    </a>

    <div class="site-header-actions">
      <nav class="nav-links" aria-label="Primary navigation">
        <a class="nav-link" href="index.html">Occupations</a>
        <a class="nav-link active" href="transitions.html">Training</a>
        <a class="nav-link" href="methodology.html">Methodology</a>
        <a class="nav-link" href="about.html">About</a>
      </nav>

      <form class="nav-search" id="globalSearchForm">
        <input
          id="searchInput"
          type="text"
          placeholder="Search all occupations"
          aria-label="Search all occupations"
        />
      </form>
    </div>
  </div>
</header>

<header class="hero compact-hero boxed-hero">
  <div class="wrap">
    <p class="eyebrow">Transition Training Pathways</p>
    <h1>AI Displacement Risk &amp; Transition Pathways</h1>
    <p class="subtitle">
        Use this tool to explore Arizona education and training programs connected to lower-exposure occupations identified through the AI Transition Pathways analysis.
        <strong>Note:</strong> Related training programs are currently available only for Medium, Low, and Very Low AI exposure occupations.
    </p>
  </div>
</header>

<div class="wrap">

    <div class="filters">

        <div>
            <label>
                Lower-Exposure Related Occupation / SOC Code
                <span class="tooltip-icon"
                    title="Select a lower-exposure occupation to view education and training programs connected through SOC-to-CIP mappings.">
                    ⓘ
                </span>
            </label>
            <select id="destinationFilter" onchange="applyFilters()">
                <option value="">Select an occupation</option>
            </select>
        </div>

        <div>
            <label>
                Search Lower-Exposure Occupation / SOC
                <span class="tooltip-icon"
                    title="Search by occupation title or SOC code to quickly find a lower-exposure occupation.">
                    ⓘ
                </span>
            </label>
            <input id="destinationSearchBox"
                   type="text"
                   placeholder="Example: Data Scientists or 15-2051"
                   oninput="updateDestinationDropdown(); applyFilters();">
        </div>

        <div>
            <label>
                Credential / Degree Type
                <span class="tooltip-icon"
                    title="Filter programs by certificate, associate degree, bachelor's degree, or other credential type.">
                    ⓘ
                </span>
            </label>
            <select id="credentialFilter" onchange="applyFilters()">
                <option value="">All</option>
            </select>
        </div>

        <div>
            <label>
                WIOA Approved
                <span class="tooltip-icon"
                    title="Filter programs that are approved for Workforce Innovation and Opportunity Act funding.">
                    ⓘ
                </span>
            </label>
            <select id="wioaFilter" onchange="applyFilters()">
                <option value="">All</option>
                <option value="Yes">Yes only</option>
            </select>
        </div>

    </div>

    <div class="summary">

        <div class="card">
            <div class="metric" id="programCount">0</div>
            <div class="metric-label">Training Programs</div>
        </div>

        <div class="card">
            <div class="metric" id="providerCount">0</div>
            <div class="metric-label">Schools / Providers</div>
        </div>

        <div class="card">
            <div class="metric" id="cipCount">0</div>
            <div class="metric-label">CIP Codes</div>
        </div>

        <div class="card">
            <div class="metric" id="wioaCount">0</div>
            <div class="metric-label">WIOA-Approved Programs</div>
        </div>

    </div>

    <div class="results-header">
        <h2>Training Programs</h2>
        <button onclick="downloadCSV()">Download Filtered CSV</button>
    </div>

    <div id="noResultsMessage" class="no-results" style="display:none;">
        No related training programs are currently available for this occupation.
    </div>

    <table id="resultsTable">

        <thead>
            <tr>
                <th>SOC Code</th>
                <th>Lower-Exposure Occupation</th>
                <th>Program</th>
                <th>School / Provider</th>
                <th>Credential</th>
                <th>Length</th>
                <th>Cost</th>
                <th>Location</th>
                <th>Delivery</th>
                <th>WIOA</th>
                <th>Website</th>
            </tr>
        </thead>

        <tbody id="resultsBody"></tbody>

    </table>

    <div class="note">
        Training programs are linked to lower-exposure occupations through SOC-to-CIP mappings.
    </div>

    <div class="note" id="displayLimitNote"></div>

</div>

<footer class="site-footer">
    <div class="wrap site-footer-inner">
        <div class="footer-brand">
            <img
            src="assets/ARIZONA%20PRIMARY%20LOGO_Solid_Agnostic_White.png"
            alt="State of Arizona"
            />
            <div>
            <h2>Arizona Office of Economic Opportunity</h2>
            <p>
                Labor market information, workforce evaluation, and economic analysis
                for the State of Arizona.
            </p>
            </div>
        </div>

        <div class="footer-links">
            <div>
            <h3>Resources</h3>
            <a href="index.html">Occupations</a>
            <a href="transitions.html">Training</a>
            <a href="methodology.html">Methodology</a>
            <a href="about.html">About</a>
            </div>

            <div>
            <h3>State Links</h3>
            <a href="https://az.gov/">AZ.gov</a>
            <a href="https://oeo.az.gov/">OEO.az.gov</a>
            <a href="https://azgovernor.gov/">Governor's Office</a>
            </div>

            <div>
            <h3>Policies</h3>
            <a href="https://az.gov/policy/accessibility">Accessibility</a>
            <a href="https://az.gov/privacy-policy">Privacy Policy</a>
            <a href="https://az.gov/policy/term-use-agreement">Terms of Use</a>
            </div>
        </div>
    </div>

    <div class="footer-bottom">
        <div class="wrap">
            <p>© State of Arizona. This site is a public-facing workforce analysis prototype.</p>
        </div>
    </div>
</footer>

<script>

const data = {data_json};

let filteredData = [];

function uniqueValues(field) {{
    return [...new Set(
        data.map(d => d[field]).filter(v => v !== "" && v !== null && v !== undefined)
    )].sort();
}}

function populateSelect(id, values, includeAll=true) {{

    const select = document.getElementById(id);

    if (!includeAll) {{
        select.innerHTML = "";
    }}

    values.forEach(v => {{
        const opt = document.createElement("option");
        opt.value = v;
        opt.textContent = v;
        select.appendChild(opt);
    }});
}}

function initFilters() {{

    const destinationSelect = document.getElementById("destinationFilter");
    destinationSelect.innerHTML = '<option value="">Select occupation</option>';

    uniqueValues("destination_display").forEach(v => {{
        const opt = document.createElement("option");
        opt.value = v;
        opt.textContent = v;
        destinationSelect.appendChild(opt);
    }});

    populateSelect(
        "credentialFilter",
        uniqueValues("credential_category")
    );

    const params = new URLSearchParams(window.location.search);
    const requestedId = params.get("id");
    const requestedSoc = params.get("soc");

    if (requestedSoc) {{
        const match = data.find(d => String(d.Destination_SOC) === requestedSoc);
        if (match) {{
            document.getElementById("destinationFilter").value = match.destination_display;
        }}
    }}

    if (requestedId) {{
        const normalizedId = requestedId.toLowerCase().trim();
        const match = data.find(d =>
            String(d.Destination_Occupation || "")
                .toLowerCase()
                .replace(/[^a-z0-9]+/g, "-")
                .replace(/^-|-$/g, "") === normalizedId
        );

        if (match) {{
            document.getElementById("destinationFilter").value = match.destination_display;
        }}
    }}
}}

function updateDestinationDropdown() {{

    const searchTerm =
        document.getElementById("destinationSearchBox")
        .value
        .toLowerCase();

    const matches = uniqueValues("destination_display").filter(v =>
        v.toLowerCase().includes(searchTerm)
    );

    populateSelect("destinationFilter", matches, false);

    if (matches.length > 0) {{
        document.getElementById("destinationFilter").value = matches[0];
    }}

    applyFilters();
}}

function applyFilters() {{

    const destination = document.getElementById("destinationFilter").value;
    const credential = document.getElementById("credentialFilter").value;
    const wioa = document.getElementById("wioaFilter").value;

    if (destination === "") {{
        filteredData = [];
        renderSummary();
        renderTable();
        return;
    }}

    filteredData = data.filter(d => {{

        return (

            (destination === "" ||
             d.destination_display === destination)

            &&

            (credential === "" ||
             d.credential_category === credential)

            &&

            (wioa === "" ||
             d.WIOA_approved === wioa)

        );
    }});

    renderSummary();
    renderTable();
}}

function renderSummary() {{

    const programData = filteredData.filter(
        d => d.has_training_program === true
    );

    const programs = new Set(
        programData.map(d =>
            d.program_name + "|" + d.provider_name
        )
    );

    const providers = new Set(
        programData.map(d => d.provider_name)
    );

    const cips = new Set(
        programData.map(d => d.CIP2020Code_Clean)
    );

    const wioaPrograms = new Set(
        programData
            .filter(d => d.WIOA_approved === "Yes")
            .map(d =>
                d.program_name + "|" + d.provider_name
            )
    );

    document.getElementById("programCount").textContent = programs.size;
    document.getElementById("providerCount").textContent = providers.size;
    document.getElementById("cipCount").textContent = cips.size;
    document.getElementById("wioaCount").textContent = wioaPrograms.size;
}}

function formatCost(value) {{

    if (value === "" || isNaN(Number(value)))
        return "";

    return "$" + Number(value).toLocaleString();
}}

function renderTable() {{

    const tbody = document.getElementById("resultsBody");
    const table = document.getElementById("resultsTable");
    const noResults = document.getElementById("noResultsMessage");

    tbody.innerHTML = "";

    const programData = filteredData.filter(
        d => d.has_training_program === true
    );

    if (programData.length === 0) {{
        table.style.display = "none";
        noResults.style.display = "block";
        return;
    }}

    table.style.display = "table";
    noResults.style.display = "none";

    const seen = new Set();

    programData.forEach(d => {{

        const key =
            d.program_name +
            "|" +
            d.provider_name +
            "|" +
            d.destination_display;

        if (seen.has(key)) return;

        seen.add(key);

        const url =
            d.program_url ||
            d.training_url ||
            "";

        const website =
            url !== ""
            ? `<a href="${{url}}" target="_blank">Open</a>`
            : "";

        const length =
            d.duration_weeks
            ? `${{d.duration_weeks}} weeks`
            : "";

        const row = `
            <tr>
                <td>${{d.Destination_SOC || ""}}</td>
                <td>${{d.Destination_Occupation || ""}}</td>
                <td>${{d.program_name || ""}}</td>
                <td>${{d.provider_name || ""}}</td>
                <td>
                    ${{d.credential_category || ""}}
                    <br>
                    <small>${{d.credential_name || ""}}</small>
                </td>
                <td>${{length}}</td>
                <td>${{formatCost(d.total_instate_cost)}}</td>
                <td>${{d.training_locations || ""}}</td>
                <td>${{d.delivery_mode || ""}}</td>
                <td>${{d.WIOA_approved || ""}}</td>
                <td>${{website}}</td>
            </tr>
        `;

        tbody.insertAdjacentHTML("beforeend", row);
    }});
}}

function downloadCSV() {{

    const programData = filteredData.filter(
        d => d.has_training_program === true
    );

    const headers = [
        "Destination_SOC",
        "Destination_Occupation",
        "program_name",
        "provider_name",
        "credential_category",
        "credential_name",
        "duration_weeks",
        "total_instate_cost",
        "training_locations",
        "delivery_mode",
        "WIOA_approved",
        "training_source"
    ];

    const rows = programData.map(d =>
        headers.map(h =>
            `"${{String(d[h] || "").replaceAll('"','""')}}"`
        ).join(",")
    );

    const csv =
        [headers.join(","), ...rows].join("\\n");

    const blob =
        new Blob([csv], {{type:"text/csv"}});

    const url =
        URL.createObjectURL(blob);

    const a =
        document.createElement("a");

    a.href = url;
    a.download = "filtered_training_programs.csv";
    a.click();

    URL.revokeObjectURL(url);
}}

document.getElementById("destinationSearchBox").addEventListener("keyup", updateDestinationDropdown);

document.querySelectorAll("select").forEach(el => {{
    el.addEventListener("change", applyFilters);
}});

initFilters();
applyFilters();

</script>

</body>
</html>
"""

# =========================================================
# SAVE HTML
# =========================================================

output_html.parent.mkdir(parents=True, exist_ok=True)
output_html.write_text(html, encoding="utf-8")

print("HTML tool saved to:")
print(output_html)

print("\\nThis version keeps occupations with 0 training programs in the dropdown.")

HTML tool saved to:
C:\Users\301533\Documents\GitHub\AI\AI Website\aipathways.github.io\transitions.html
\nThis version keeps occupations with 0 training programs in the dropdown.


In [ ]:
# import pandas as pd
# import json
# from pathlib import Path

# # =========================================================
# # CREATE HTML TRAINING PATHWAYS TOOL
# # =========================================================

# folder = Path(r"C:\Users\301533\Documents\AI Analysis\Training Programs (ETPL+IPEDS)")

# input_file = folder / "TRANSITION_TRAINING_PROGRAMS_MASTER.csv"
# output_html = folder / "AI_Training_Pathways_Tool.html"

# # =========================================================
# # LOAD DATA
# # =========================================================

# df = pd.read_csv(input_file, low_memory=False)

# df["has_training_program"] = (
#     df["has_training_program"]
#       .astype(str)
#       .str.lower()
#       .isin(["true", "1", "yes"])
# )

# df["training_program_count"] = df.groupby(
#     ["Destination_Occupation", "Destination_SOC"]
# )["program_name"].transform(lambda x: x.notna().sum())

# # =========================================================
# # KEEP ONLY NEEDED COLUMNS
# # =========================================================

# keep_cols = [
#     "Destination_Occupation",
#     "Destination_SOC",
#     "Transition_Type",
#     "has_training_program",
#     "training_program_count",
#     "training_source",
#     "program_name",
#     "provider_name",
#     "credential_category",
#     "credential_name",
#     "delivery_mode",
#     "duration_weeks",
#     "duration_hours",
#     "total_instate_cost",
#     "training_locations",
#     "WIOA_approved",
#     "program_url",
#     "training_url",
#     "CIP2020Code_Clean"
# ]

# df = df[[c for c in keep_cols if c in df.columns]].copy()

# # Fill blanks
# df = df.fillna("")

# # Drop duplicates
# df = df.drop_duplicates()

# # =========================================================
# # CREATE DISPLAY OCCUPATION
# # =========================================================

# df["destination_display"] = (
#     df["Destination_Occupation"].astype(str)
#     + " ("
#     + df["Destination_SOC"].astype(str)
#     + ")"
# )

# # =========================================================
# # CONVERT TO JSON
# # =========================================================

# records = df.to_dict(orient="records")
# data_json = json.dumps(records, ensure_ascii=False)

# # =========================================================
# # BUILD HTML
# # =========================================================

# html = f"""
# <!DOCTYPE html>
# <html lang="en">
# <head>
# <meta charset="UTF-8">
# <title>AI Transition Training Pathways</title>
# <meta name="viewport" content="width=device-width, initial-scale=1.0">

# <style>

# body {{
#     font-family: Arial, sans-serif;
#     margin: 0;
#     background: #f5f7fa;
#     color: #1f2937;
# }}

# header {{
#     background: #2c5d7c;
#     color: white;
#     padding: 20px 24px 40px 24px;
# }}

# header h1 {{
#     margin: 0;
#     font-size: 28px;
# }}

# header p {{
#     margin-top: 10px;
#     font-size: 15px;
#     max-width: 900px;
#     line-height: 1.5;
# }}

# .container {{
#     padding: 28px 22px;
# }}

# .filters {{
#     display: grid;
#     grid-template-columns: repeat(3, 1fr);
#     gap: 18px;
#     background: white;
#     padding: 20px;
#     border-radius: 16px;
#     box-shadow: 0 2px 10px rgba(0,0,0,0.06);
# }}

# label {{
#     font-weight: bold;
#     display: block;
#     margin-bottom: 8px;
#     font-size: 14px;
# }}

# select, input {{
#     width: 100%;
#     box-sizing: border-box;
#     padding: 10px;
#     border-radius: 8px;
#     border: 1px solid #d4d9df;
#     font-size: 14px;
# }}

# .summary {{
#     display: grid;
#     grid-template-columns: repeat(4, 1fr);
#     gap: 16px;
#     margin-top: 22px;
#     margin-bottom: 26px;
# }}

# .card {{
#     background: white;
#     border-radius: 16px;
#     padding: 22px;
#     box-shadow: 0 2px 10px rgba(0,0,0,0.06);
# }}

# .metric {{
#     font-size: 24px;
#     font-weight: bold;
#     color: #2c5d7c;
# }}

# .metric-label {{
#     margin-top: 6px;
#     color: #667085;
# }}

# .results-header {{
#     display: flex;
#     justify-content: space-between;
#     align-items: center;
#     margin-bottom: 14px;
# }}

# button {{
#     background: #5bc7b8;
#     color: white;
#     border: none;
#     padding: 12px 16px;
#     border-radius: 8px;
#     font-weight: bold;
#     cursor: pointer;
# }}

# table {{
#     width: 100%;
#     border-collapse: collapse;
#     background: white;
#     border-radius: 16px;
#     overflow: hidden;
#     box-shadow: 0 2px 10px rgba(0,0,0,0.06);
# }}

# th {{
#     background: #2c5d7c;
#     color: white;
#     text-align: left;
#     padding: 14px 10px;
#     font-size: 13px;
# }}

# td {{
#     padding: 12px 10px;
#     border-bottom: 1px solid #e7ebf0;
#     vertical-align: top;
#     font-size: 13px;
# }}

# tr:hover {{
#     background: #f3fbfa;
# }}

# a {{
#     color: #2c5d7c;
#     font-weight: bold;
# }}

# .note {{
#     margin-top: 16px;
#     font-size: 13px;
#     color: #667085;
# }}

# @media (max-width: 1000px) {{

#     .filters {{
#         grid-template-columns: 1fr;
#     }}

#     .summary {{
#         grid-template-columns: 1fr;
#     }}

# }}

# </style>
# </head>

# <body>

# <header>
#     <h1>AI Transition Training Pathways</h1>

#     <p>
#         Select a lower-exposure related occupation to view Arizona training programs connected through SOC-to-CIP mappings.
#     </p>
# </header>

# <div class="container">

#     <div class="filters">

#         <div>
#             <label>Lower-Exposure Related Occupation / SOC Code</label>
#             <select id="destinationFilter"></select>
#         </div>

#         <div>
#             <label>Search Lower-Exposure Occupation / SOC</label>
#             <input id="destinationSearchBox"
#                    type="text"
#                    placeholder="Example: Data Scientists or 15-2051">
#         </div>

#         <div>
#             <label>Pathway Type</label>
#             <select id="pathwayFilter">
#                 <option value="">All</option>
#             </select>
#         </div>

#         <div>
#             <label>Training Source</label>
#             <select id="sourceFilter">
#                 <option value="">All</option>
#             </select>
#         </div>

#         <div>
#             <label>Credential / Degree Type</label>
#             <select id="credentialFilter">
#                 <option value="">All</option>
#             </select>
#         </div>

#         <div>
#             <label>WIOA Approved</label>
#             <select id="wioaFilter">
#                 <option value="">All</option>
#                 <option value="Yes">Yes only</option>
#             </select>
#         </div>

#     </div>

#     <div class="summary">

#         <div class="card">
#             <div class="metric" id="programCount">0</div>
#             <div class="metric-label">Training Programs</div>
#         </div>

#         <div class="card">
#             <div class="metric" id="providerCount">0</div>
#             <div class="metric-label">Schools / Providers</div>
#         </div>

#         <div class="card">
#             <div class="metric" id="cipCount">0</div>
#             <div class="metric-label">CIP Codes</div>
#         </div>

#         <div class="card">
#             <div class="metric" id="wioaCount">0</div>
#             <div class="metric-label">WIOA-Approved Programs</div>
#         </div>

#     </div>

#     <div class="results-header">
#         <h2>Training Programs</h2>
#         <button onclick="downloadCSV()">Download Filtered CSV</button>
#     </div>

#     <table>

#         <thead>
#             <tr>
#                 <th>SOC Code</th>
#                 <th>Lower-Exposure Occupation</th>
#                 <th>Program</th>
#                 <th>School / Provider</th>
#                 <th>Credential</th>
#                 <th>Length</th>
#                 <th>Cost</th>
#                 <th>Location</th>
#                 <th>Delivery</th>
#                 <th>WIOA</th>
#                 <th>Source</th>
#                 <th>Website</th>
#             </tr>
#         </thead>

#         <tbody id="resultsBody"></tbody>

#     </table>

#     <div class="note">
#         Training programs are linked to lower-exposure occupations through SOC-to-CIP mappings.
#     </div>

# </div>

# <script>

# const data = {data_json};

# let filteredData = [];

# function uniqueValues(field) {{
#     return [...new Set(
#         data.map(d => d[field]).filter(v => v !== "")
#     )].sort();
# }}

# function populateSelect(id, values, includeAll=true) {{

#     const select = document.getElementById(id);

#     if (!includeAll) {{
#         select.innerHTML = "";
#     }}

#     values.forEach(v => {{
#         const opt = document.createElement("option");
#         opt.value = v;
#         opt.textContent = v;
#         select.appendChild(opt);
#     }});
# }}

# function initFilters() {{

#     const destinationSelect = document.getElementById("destinationFilter");
#     destinationSelect.innerHTML = '<option value="">Select occupation</option>';

#     uniqueValues("destination_display").forEach(v => {{
#         const opt = document.createElement("option");
#         opt.value = v;
#         opt.textContent = v;
#         destinationSelect.appendChild(opt);
#     }});

#     populateSelect(
#         "pathwayFilter",
#         uniqueValues("Transition_Type")
#     );

#     populateSelect(
#         "sourceFilter",
#         uniqueValues("training_source")
#     );

#     populateSelect(
#         "credentialFilter",
#         uniqueValues("credential_category")
#     );
# }}

# function updateDestinationDropdown() {{

#     const searchTerm =
#         document.getElementById("destinationSearchBox")
#         .value
#         .toLowerCase();

#     const matches = uniqueValues("destination_display").filter(v =>
#         v.toLowerCase().includes(searchTerm)
#     );

#     populateSelect("destinationFilter", matches, false);

#     if (matches.length > 0) {{
#         document.getElementById("destinationFilter").value = matches[0];
#     }}

#     applyFilters();
# }}

# function applyFilters() {{

#     const destination = document.getElementById("destinationFilter").value;
#     const destinationSearch =
#         document.getElementById("destinationSearchBox")
#         .value
#         .toLowerCase();

#     const pathway = document.getElementById("pathwayFilter").value;
#     const source = document.getElementById("sourceFilter").value;
#     const credential = document.getElementById("credentialFilter").value;
#     const wioa = document.getElementById("wioaFilter").value;

#     if (destination === "") {{
#         filteredData = [];
#         renderSummary();
#         renderTable();
#         return;
#     }}

#     filteredData = data.filter(d => {{

#         const destinationText =
#             (d.destination_display || "").toLowerCase();

#         return (

#             (destination === "" ||
#              d.destination_display === destination)

#             &&

#             (pathway === "" ||
#             d["Transition_Type"] === pathway)

#             &&

#             (source === "" ||
#              d.training_source === source)

#             &&

#             (credential === "" ||
#              d.credential_category === credential)

#             &&

#             (wioa === "" ||
#              d.WIOA_approved === wioa)

#         );
#     }});

#     renderSummary();
#     renderTable();
# }}

# function renderSummary() {{

#     const programData = filteredData.filter(
#         d => d.has_training_program === true
#     );

#     const programs = new Set(
#         programData.map(d =>
#             d.program_name + "|" + d.provider_name
#         )
#     );

#     const providers = new Set(
#         programData.map(d => d.provider_name)
#     );

#     const cips = new Set(
#         programData.map(d => d.CIP2020Code_Clean)
#     );

#     const wioaPrograms = new Set(
#         programData
#             .filter(d => d.WIOA_approved === "Yes")
#             .map(d =>
#                 d.program_name + "|" + d.provider_name
#             )
#     );

#     document.getElementById("programCount").textContent =
#         programs.size;

#     document.getElementById("providerCount").textContent =
#         providers.size;

#     document.getElementById("cipCount").textContent =
#         cips.size;

#     document.getElementById("wioaCount").textContent =
#         wioaPrograms.size;
# }}

# function formatCost(value) {{

#     if (value === "" || isNaN(Number(value)))
#         return "";

#     return "$" + Number(value).toLocaleString();
# }}

# function renderTable() {{

#     const tbody = document.getElementById("resultsBody");

#     tbody.innerHTML = "";

#     const seen = new Set();

#     const tableData = filteredData.filter(
#     d => d.has_training_program === true
#     );

#     tableData.forEach(d => {{

#         const key =
#             d.program_name +
#             "|" +
#             d.provider_name +
#             "|" +
#             d.destination_display;

#         if (seen.has(key)) return;

#         seen.add(key);

#         const url =
#             d.program_url ||
#             d.training_url ||
#             "";

#         const website =
#             url !== ""
#             ? `<a href="${{url}}" target="_blank">Open</a>`
#             : "";

#         const length =
#             d.duration_weeks
#             ? `${{d.duration_weeks}} weeks`
#             : "";

#         const row = `
#             <tr>

#                 <td>${{d.Destination_SOC || ""}}</td>

#                 <td>${{d.Destination_Occupation || ""}}</td>

#                 <td>${{d.program_name || ""}}</td>

#                 <td>${{d.provider_name || ""}}</td>

#                 <td>
#                     ${{d.credential_category || ""}}
#                     <br>
#                     <small>${{d.credential_name || ""}}</small>
#                 </td>

#                 <td>${{length}}</td>

#                 <td>${{formatCost(d.total_instate_cost)}}</td>

#                 <td>${{d.training_locations || ""}}</td>

#                 <td>${{d.delivery_mode || ""}}</td>

#                 <td>${{d.WIOA_approved || ""}}</td>

#                 <td>${{d.training_source || ""}}</td>

#                 <td>${{website}}</td>

#             </tr>
#         `;

#         tbody.insertAdjacentHTML("beforeend", row);
#     }});
# }}

# function downloadCSV() {{

#     const headers = [
#         "Destination_SOC",
#         "Destination_Occupation",
#         "program_name",
#         "provider_name",
#         "credential_category",
#         "credential_name",
#         "duration_weeks",
#         "total_instate_cost",
#         "training_locations",
#         "delivery_mode",
#         "WIOA_approved",
#         "training_source"
#     ];

#     const rows = filteredData.map(d =>
#         headers.map(h =>
#             `"${{String(d[h] || "").replaceAll('"','""')}}"`
#         ).join(",")
#     );

#     const csv =
#         [headers.join(","), ...rows].join("\\n");

#     const blob =
#         new Blob([csv], {{type:"text/csv"}});

#     const url =
#         URL.createObjectURL(blob);

#     const a =
#         document.createElement("a");

#     a.href = url;

#     a.download =
#         "filtered_training_programs.csv";

#     a.click();

#     URL.revokeObjectURL(url);
# }}

# document.getElementById("destinationSearchBox").addEventListener("keyup", updateDestinationDropdown);

# document.querySelectorAll("select").forEach(el => {{
#     el.addEventListener("change", applyFilters);
# }});

# initFilters();
# applyFilters();

# </script>

# </body>
# </html>
# """

# # =========================================================
# # SAVE HTML
# # =========================================================

# output_html.write_text(html, encoding="utf-8")

# print("HTML tool saved to:")
# print(output_html)

# print("\\nDouble-click the HTML file to open it in Chrome.")


HTML tool saved to:
C:\Users\301533\Documents\AI Analysis\Training Programs (ETPL+IPEDS)\AI_Training_Pathways_Tool.html
\nDouble-click the HTML file to open it in Chrome.


In [ ]:
import pandas as pd
from pathlib import Path

# Original file
input_file = Path(
    r"G:\.shortcut-targets-by-id\1KKrBsNyhwzbYus9ExtUkUd_3jE4VZ4W6\AI_Analysis\Arielle's Stuff\High_VeryHigh_AND_Medium_Related_Occs_Final.xlsx"
)

# New cleaned file
output_file = input_file.with_name(
    "High_VeryHigh_AND_Medium_Related_Occs_Final_CLEANED_SOC.xlsx"
)

# Load
df = pd.read_excel(input_file, dtype=object)

soc_cols = ["Source_SOC", "Gateway_SOC", "Target_SOC", "Destination_SOC"]

def fix_soc(value):
    if pd.isna(value):
        return ""

    # If Excel converted SOC to datetime
    if isinstance(value, pd.Timestamp):
        return f"'{value.month:02d}-{value.year}"

    text = str(value).strip()

    # If pandas reads it as datetime string
    dt = pd.to_datetime(text, errors="coerce")
    if pd.notna(dt) and ("00:00:00" in text or "-" in text):
        # Only fix if it looks like a date-converted SOC
        if dt.month == 11:
            return f"'11-{dt.year}"

    # If Google/Excel shows Nov-21, Nov-11, etc.
    if text.startswith("Nov-"):
        year_part = text.replace("Nov-", "").strip()
        if len(year_part) == 2:
            year_part = "20" + year_part
        return f"'11-{year_part}"

    return text

# Apply cleanup
for col in soc_cols:
    df[col] = df[col].apply(fix_soc)

# Save separate cleaned file
df.to_excel(output_file, index=False)

print("Saved cleaned file to:")
print(output_file)

Saved cleaned file to:
G:\.shortcut-targets-by-id\1KKrBsNyhwzbYus9ExtUkUd_3jE4VZ4W6\AI_Analysis\Arielle's Stuff\High_VeryHigh_AND_Medium_Related_Occs_Final_CLEANED_SOC.xlsx
